In [1]:
import torch
from mamba_ssm import Mamba

model = Mamba(
    d_model=128,
    d_state=16,
    d_conv=4,
    expand=2
).cuda()

x = torch.randn(1, 64, 128).cuda()
y = model(x)

print("Success:", y.shape)

Success: torch.Size([1, 64, 128])


In [2]:
"""
Mamba-UNet++ v4 — DFUC2022 Edition  (50-epoch starter, full ablation suite)
=============================================================================

New in this version
────────────────────
  A. EPOCHS = 50  (quick first run; change at top)
  B. PATIENCE = 50  (less aggressive — won't fire before epoch 50)
  C. Computational Complexity Analysis
       • Trainable params
       • GFLOPs  (via torchinfo if available, else ptflops fallback)
       • FPS / mean inference latency  (GPU timed, single image)
  D. Auto Metric Saving
       • metrics_history.csv   — per-epoch Dice/IoU/Prec/Rec/F1/LR
       • run_summary.json      — best scores + complexity + ablation tag
  E. Ablation Suite  (set ABLATION at the top)
       "none"            — full model  (default)
       "no_mamba"        — replace Mamba with Identity
       "no_deep_sup"     — final output only, no deep supervision
       "no_bidirectional"— forward scan only in FastMambaBlock
       "dice_bce_only"   — Dice+BCE loss, skip Tversky & Lovász
"""

# ── Thread guards MUST come before any cv2 / numpy import ────────────
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# =====================================================================
# ★  MASTER CONFIGURATION — edit here only  ★
# =====================================================================
EPOCHS    = 50          # change to 150 for full run
PATIENCE  = 50          # early-stop patience (epochs without improvement)
ABLATION  = "none"      # "none" | "no_mamba" | "no_deep_sup"
                        # "no_bidirectional" | "dice_bce_only"

TRAIN_PATCH_SIZE = 512
INFER_PATCH_SIZE = 512
INFER_STRIDE     = 384
PAD_RESIZE       = 640
IMG_SIZE         = TRAIN_PATCH_SIZE

# =====================================================================

import csv
import json
import math
import random
import time
import warnings
from copy import deepcopy

import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── optional fast Mamba ──────────────────────────────────────────────
try:
    from mamba_ssm import Mamba
    USE_FAST_MAMBA = True
    print("✓ Fast CUDA Mamba loaded")
except ImportError:
    USE_FAST_MAMBA = False
    print("⚠  mamba-ssm not found — using CBAM attention (fast fallback)")

# ── optional timm ────────────────────────────────────────────────────
try:
    import timm
    HAS_TIMM = True
    print("✓ timm loaded — will use pretrained EfficientNet-B4 encoder")
except ImportError:
    HAS_TIMM = False
    print("⚠  timm not found — falling back to scratch encoder")

# ── complexity tools (graceful fallback) ────────────────────────────
try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except ImportError:
    HAS_TORCHINFO = False

try:
    from ptflops import get_model_complexity_info
    HAS_PTFLOPS = True
except ImportError:
    HAS_PTFLOPS = False

# Active-ablation tag — appended to all saved filenames
ABL_TAG = f"abl_{ABLATION}" if ABLATION != "none" else "full"

# =====================================================================
# 0. SEED
# =====================================================================
def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()

# =====================================================================
# 1. HELPERS — Aspect-ratio-preserving pad-to-square
# =====================================================================
def pad_to_square(image, mask=None, pad_value=0):
    h, w = image.shape[:2]
    if h == w:
        if mask is not None:
            return image, mask, (0, 0, 0, 0)
        return image, (0, 0, 0, 0)

    diff = abs(h - w)
    pad_before = diff // 2
    pad_after  = diff - pad_before

    if h > w:
        top, bottom, left, right = 0, 0, pad_before, pad_after
    else:
        top, bottom, left, right = pad_before, pad_after, 0, 0

    image = cv2.copyMakeBorder(
        image, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=(pad_value, pad_value, pad_value),
    )
    if mask is not None:
        mask = cv2.copyMakeBorder(
            mask, top, bottom, left, right,
            cv2.BORDER_CONSTANT, value=0,
        )
        return image, mask, (top, bottom, left, right)
    return image, (top, bottom, left, right)


def unpad(arr, pad_info, orig_h, orig_w):
    top, bottom, left, right = pad_info
    h, w = arr.shape[:2]
    arr = arr[
        top : h - bottom if bottom else h,
        left : w - right if right else w,
    ]
    if arr.shape[0] != orig_h or arr.shape[1] != orig_w:
        arr = cv2.resize(
            arr.astype(np.float32), (orig_w, orig_h),
            interpolation=cv2.INTER_LINEAR,
        )
    return arr

# =====================================================================
# 2. DATASET
# =====================================================================
class DFUC2022Dataset(Dataset):
    def __init__(
        self, image_dir, mask_dir, names,
        transform=None, training=True,
        patch_size=TRAIN_PATCH_SIZE, crops_per_image=1,
    ):
        self.transform       = transform
        self.training        = training
        self.patch_size      = patch_size
        self.crops_per_image = crops_per_image
        self.samples         = []

        for n in names:
            ip = os.path.join(image_dir, n)
            if not os.path.exists(ip) or os.path.getsize(ip) == 0:
                continue
            stem   = n.rsplit(".", 1)[0]
            mp_png = os.path.join(mask_dir, stem + ".png")
            mp_jpg = os.path.join(mask_dir, n)
            mp = (mp_png if os.path.exists(mp_png)
                  else (mp_jpg if os.path.exists(mp_jpg) else None))
            self.samples.append({"img": ip, "mask": mp})

    def __len__(self):
        return len(self.samples) * self.crops_per_image if self.training else len(self.samples)

    @staticmethod
    def _bbox_of_mask(mask):
        ys, xs = np.where(mask > 0.5)
        if len(ys) == 0:
            return None
        return int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())

    def _random_crop_around_fg(self, image, mask):
        ps   = self.patch_size
        h, w = image.shape[:2]

        pad_h = max(ps - h, 0)
        pad_w = max(ps - w, 0)
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w,
                                       cv2.BORDER_CONSTANT, value=(0, 0, 0))
            mask  = cv2.copyMakeBorder(mask,  0, pad_h, 0, pad_w,
                                       cv2.BORDER_CONSTANT, value=0)
            h, w  = image.shape[:2]

        bbox = self._bbox_of_mask(mask)
        if bbox is not None:
            y0, y1, x0, x1 = bbox
            cy_lo = max(0, y1 - ps + 1);  cy_hi = min(h - ps, y0)
            cx_lo = max(0, x1 - ps + 1);  cx_hi = min(w - ps, x0)
            cy = random.randint(min(cy_lo, cy_hi), max(cy_lo, cy_hi))
            cx = random.randint(min(cx_lo, cx_hi), max(cx_lo, cx_hi))
        else:
            cy = random.randint(0, h - ps)
            cx = random.randint(0, w - ps)

        return image[cy:cy+ps, cx:cx+ps], mask[cy:cy+ps, cx:cx+ps]

    def __getitem__(self, idx):
        real_idx = idx % len(self.samples)
        s = self.samples[real_idx]

        im = cv2.imread(s["img"])
        im = (cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
              if im is not None
              else np.zeros((TRAIN_PATCH_SIZE, TRAIN_PATCH_SIZE, 3), np.uint8))
        mk = (cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
              if s["mask"] else np.zeros(im.shape[:2], np.uint8))
        mk = (mk > 127).astype(np.float32)

        if self.training:
            im, mk = self._random_crop_around_fg(im, mk)
        else:
            im, mk, _ = pad_to_square(im, mk, pad_value=0)
            im = cv2.resize(im, (PAD_RESIZE, PAD_RESIZE))
            mk = cv2.resize(mk, (PAD_RESIZE, PAD_RESIZE),
                            interpolation=cv2.INTER_NEAREST)

        if self.transform:
            aug = self.transform(image=im, mask=mk)
            im, mk = aug["image"], aug["mask"].unsqueeze(0)
        return im, mk


def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08),
                 rotate=(-30, 30), p=0.5),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05, p=0.2),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
        A.HueSaturationValue(8, 20, 10, p=0.3),
        A.GaussNoise(var_limit=(5, 20), p=0.15),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# =====================================================================
# 3. METRICS
# =====================================================================
def calc_metrics(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float().cpu().numpy()
    tgt  = target.float().cpu().numpy()
    dice_l, iou_l, prec_l, rec_l, f1_l = [], [], [], [], []
    for p, t in zip(pred, tgt):
        p, t = p.squeeze(), t.squeeze()
        tp = float((p * t).sum())
        fp = float((p * (1 - t)).sum())
        fn = float(((1 - p) * t).sum())
        d  = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
        i  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
        pr = (tp + 1e-6) / (tp + fp + 1e-6)
        rc = (tp + 1e-6) / (tp + fn + 1e-6)
        f1 = (2 * pr * rc + 1e-6) / (pr + rc + 1e-6)
        dice_l.append(d); iou_l.append(i)
        prec_l.append(pr); rec_l.append(rc); f1_l.append(f1)
    return list(map(np.mean, [dice_l, iou_l, prec_l, rec_l, f1_l]))


def hausdorff_95(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float().cpu().numpy()
    tgt  = target.float().cpu().numpy()
    hd_l = []
    for p, t in zip(pred, tgt):
        p, t = p.squeeze().astype(bool), t.squeeze().astype(bool)
        if not p.any() or not t.any():
            hd_l.append(float(np.sqrt(p.shape[0]**2 + p.shape[1]**2)))
            continue
        d1 = distance_transform_edt(~p)[t]
        d2 = distance_transform_edt(~t)[p]
        hd_l.append(np.percentile(np.concatenate([d1, d2]), 95))
    return np.mean(hd_l)

# =====================================================================
# 4. LOSSES
# =====================================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        logits, targets = logits.float(), targets.float()
        bce   = self.bce(logits, targets)
        p     = torch.sigmoid(logits).view(-1)
        t     = targets.view(-1)
        inter = (p * t).sum()
        dice  = 1 - (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce, dice


class LovaszHingeLoss(nn.Module):
    def forward(self, logits, targets):
        logits, targets = logits.float(), targets.float()
        probs = torch.sigmoid(logits).view(-1)
        tgts  = targets.view(-1)
        if tgts.sum() == 0 or tgts.sum() == tgts.numel():
            return (logits * 0.0).mean()
        errors        = (tgts - probs).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        gt_sorted = tgts[perm]
        grad      = self._lovasz_grad(gt_sorted)
        return torch.clamp(torch.dot(errors_sorted, grad), 0.0, 2.0)

    @staticmethod
    def _lovasz_grad(gt_sorted):
        gts          = gt_sorted.sum()
        intersection = gts - gt_sorted.cumsum(0)
        union        = gts + (1 - gt_sorted).cumsum(0)
        jaccard      = 1.0 - intersection / (union + 1e-6)
        if gt_sorted.numel() > 1:
            jaccard[1:] = jaccard[1:] - jaccard[:-1]
        return jaccard


class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-3):
        super().__init__()
        self.alpha = alpha; self.beta = beta
        self.gamma = gamma; self.smooth = smooth

    def forward(self, inputs, targets):
        p  = torch.sigmoid(inputs.float()).view(-1)
        t  = targets.float().view(-1)
        tp = (p * t).sum()
        fp = (p * (1 - t)).sum()
        fn = ((1 - p) * t).sum()
        tv = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth)
        return (1 - tv) ** self.gamma


class ScheduledLoss(nn.Module):
    """Full schedule — automatically reduced to Dice+BCE if ablation='dice_bce_only'."""

    def __init__(self, dice_bce_only=False):
        super().__init__()
        self.dice_bce_only = dice_bce_only
        self.dice_bce = DiceBCELoss()
        self.tversky  = TverskyLoss(alpha=0.7, beta=0.3, gamma=0.75)
        self.lovasz   = LovaszHingeLoss()
        self._epoch   = 0

    def set_epoch(self, e):
        self._epoch = e

    def forward(self, logits, targets):
        bce, dice = self.dice_bce(logits, targets)

        def safe(t):
            return t if torch.isfinite(t) else torch.zeros(1, device=logits.device)[0]

        if self.dice_bce_only:
            return 0.5 * safe(bce) + 0.5 * safe(dice)

        tv  = self.tversky(logits, targets)
        lov = self.lovasz(logits, targets)

        e = self._epoch
        if   e < 20: w = (0.25, 0.25, 0.40, 0.10)
        elif e < 50: w = (0.20, 0.30, 0.30, 0.20)
        elif e < 90: w = (0.15, 0.30, 0.25, 0.30)
        else:        w = (0.10, 0.30, 0.20, 0.40)

        return (w[0]*safe(bce) + w[1]*safe(dice)
                + w[2]*safe(tv) + w[3]*safe(lov))

# =====================================================================
# 5. BUILDING BLOCKS
# =====================================================================
class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True),
            nn.Linear(c // r, c, bias=False), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.se(x).view(x.size(0), x.size(1), 1, 1)


class CBAM(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True),
            nn.Linear(c // r, c, bias=False),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.BatchNorm2d(1), nn.Sigmoid(),
        )

    def forward(self, x):
        avg = x.mean(dim=(2, 3));  mx = x.amax(dim=(2, 3))
        ca  = (torch.sigmoid(self.mlp(avg) + self.mlp(mx))
               .unsqueeze(-1).unsqueeze(-1))
        x   = x * ca
        sa  = torch.cat([x.mean(dim=1, keepdim=True),
                         x.amax(dim=1, keepdim=True)], dim=1)
        return x * self.spatial(sa)


class IdentityBlock(nn.Module):
    """No-op replacement used in 'no_mamba' ablation."""
    def __init__(self, c):
        super().__init__()

    def forward(self, x):
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, drop=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.GELU(), nn.Dropout2d(drop),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.GELU(),
        )
        self.se   = SEBlock(out_c)
        self.proj = (nn.Conv2d(in_c, out_c, 1, bias=False)
                     if in_c != out_c else nn.Identity())

    def forward(self, x):
        return self.se(self.conv(x)) + self.proj(x)


class ASPP(nn.Module):
    def __init__(self, in_c, out_c=256):
        super().__init__()
        rates = [1, 6, 12, 18]
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_c, out_c,
                          3 if r > 1 else 1,
                          padding=r if r > 1 else 0,
                          dilation=r, bias=False),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            ) for r in rates
        ])
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
        self.proj = nn.Sequential(
            nn.Conv2d(out_c * (len(rates) + 1), out_c, 1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Dropout2d(0.1),
        )

    def forward(self, x):
        h, w  = x.shape[2:]
        feats = [c(x) for c in self.convs]
        feats.append(F.interpolate(self.pool(x), size=(h, w),
                                   mode="bilinear", align_corners=True))
        return self.proj(torch.cat(feats, 1))


class AttentionGate(nn.Module):
    def __init__(self, Fg, Fl, Fi):
        super().__init__()
        self.Wg  = nn.Sequential(nn.Conv2d(Fg, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.Wx  = nn.Sequential(nn.Conv2d(Fl, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi, 1, 1, bias=False),
                                 nn.BatchNorm2d(1), nn.Sigmoid())

    def forward(self, g, x):
        g1  = self.Wg(F.interpolate(g, x.shape[2:], mode="bilinear", align_corners=True))
        out = self.psi(F.relu(g1 + self.Wx(x), inplace=True))
        return x * out


class FastMambaBlock(nn.Module):
    def __init__(self, d, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.mamba = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.norm  = nn.LayerNorm(d)

    def _scan(self, x, flip=False):
        s = torch.flip(x, [1]) if flip else x
        o = self.mamba(self.norm(s))
        return torch.flip(o, [1]) if flip else o

    def forward(self, x):
        B, C, H, W = x.shape
        h  = x.view(B, C, -1).transpose(1, 2)

        if self.bidirectional:
            oh = (self._scan(h) + self._scan(h, flip=True)) / 2
        else:
            oh = self._scan(h)       # forward only

        v  = x.permute(0, 1, 3, 2).contiguous().view(B, C, -1).transpose(1, 2)

        if self.bidirectional:
            ov = (self._scan(v) + self._scan(v, flip=True)) / 2
        else:
            ov = self._scan(v)

        out = (oh + ov) / 2
        return out.transpose(1, 2).view(B, C, H, W) + x

# =====================================================================
# 6. ENCODER
# =====================================================================
class ResidualAdapter(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.needs_proj = in_c != out_c
        if self.needs_proj:
            self.proj = nn.Conv2d(in_c, out_c, 1, bias=False)
            self.bn   = nn.BatchNorm2d(out_c)
            nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")

    def forward(self, x):
        return self.bn(self.proj(x)) if self.needs_proj else x


class EfficientNetEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.stages   = timm.create_model("efficientnet_b4",
                                          pretrained=pretrained,
                                          features_only=True)
        self.channels = self.stages.feature_info.channels()
        print(f"  EfficientNet-B4 encoder channels: {self.channels}")

    def forward(self, x):
        return self.stages(x)


class ScratchEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        F_ = [32, 64, 128, 256, 512]
        self.channels = F_
        self.s0 = ConvBlock(3, F_[0])
        self.s1 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(F_[0], F_[1]))
        self.s2 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(F_[1], F_[2]))
        self.s3 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(F_[2], F_[3]), CBAM(F_[3]))
        self.s4 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(F_[3], F_[4]), CBAM(F_[4]))

    def forward(self, x):
        f0 = self.s0(x);  f1 = self.s1(f0);  f2 = self.s2(f1)
        f3 = self.s3(f2); f4 = self.s4(f3)
        return [f0, f1, f2, f3, f4]

# =====================================================================
# 7. MAMBA UNET++ V4  (ablation-aware)
# =====================================================================
class MambaUNetPPv4(nn.Module):
    def __init__(
        self, num_classes=1, deep_supervision=True,
        pretrained=True, ablation="none",
    ):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.ablation         = ablation

        # ── context block selection ───────────────────────────────────
        if ablation == "no_mamba":
            def ContextBlock(c): return IdentityBlock(c)
        elif USE_FAST_MAMBA and ablation != "no_mamba":
            def ContextBlock(c):
                bidi = (ablation != "no_bidirectional")
                return FastMambaBlock(c, bidirectional=bidi)
        else:
            def ContextBlock(c): return CBAM(c)

        if HAS_TIMM:
            self.encoder = EfficientNetEncoder(pretrained=pretrained)
        else:
            self.encoder = ScratchEncoder()

        enc_ch = self.encoder.channels
        dec_ch = [64, 128, 192, 256, 256]

        self.adapt = nn.ModuleList(
            [ResidualAdapter(ec, dc) for ec, dc in zip(enc_ch, dec_ch)])

        self.aspp = ASPP(dec_ch[4], dec_ch[4])
        self.ctx3 = ContextBlock(dec_ch[3])
        self.ctx4 = ContextBlock(dec_ch[4])

        C = dec_ch
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.ag = nn.ModuleDict({
            "01": AttentionGate(C[1], C[0], C[0]//2),
            "11": AttentionGate(C[2], C[1], C[1]//2),
            "21": AttentionGate(C[3], C[2], C[2]//2),
            "31": AttentionGate(C[4], C[3], C[3]//2),
            "02": AttentionGate(C[1], C[0], C[0]//2),
            "12": AttentionGate(C[2], C[1], C[1]//2),
            "22": AttentionGate(C[3], C[2], C[2]//2),
            "03": AttentionGate(C[1], C[0], C[0]//2),
            "13": AttentionGate(C[2], C[1], C[1]//2),
            "04": AttentionGate(C[1], C[0], C[0]//2),
        })

        self.d01 = ConvBlock(C[0] + C[1],        C[0])
        self.d11 = ConvBlock(C[1] + C[2],        C[1])
        self.d21 = ConvBlock(C[2] + C[3],        C[2])
        self.d31 = ConvBlock(C[3] + C[4],        C[3])
        self.d02 = ConvBlock(C[0]*2 + C[1],      C[0])
        self.d12 = ConvBlock(C[1]*2 + C[2],      C[1])
        self.d22 = ConvBlock(C[2]*2 + C[3],      C[2])
        self.d03 = ConvBlock(C[0]*3 + C[1],      C[0])
        self.d13 = ConvBlock(C[1]*3 + C[2],      C[1])
        self.d04 = ConvBlock(C[0]*4 + C[1],      C[0])

        if deep_supervision:
            self.heads = nn.ModuleList(
                [nn.Conv2d(C[0], num_classes, 1) for _ in range(4)])
        else:
            self.head = nn.Conv2d(C[0], num_classes, 1)

        self._init_decoder_weights()

    def _init_decoder_weights(self):
        enc_mods = set(id(m) for m in self.encoder.modules())
        for m in self.modules():
            if id(m) in enc_mods:
                continue
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        feats = self.encoder(x)
        f     = [ad(fe) for ad, fe in zip(self.adapt, feats)]
        f[3]  = self.ctx3(f[3])
        f[4]  = self.ctx4(self.aspp(f[4]))

        x01 = self.d01(torch.cat([self.ag["01"](f[1], f[0]), self.up(f[1])],  1))
        x11 = self.d11(torch.cat([self.ag["11"](f[2], f[1]), self.up(f[2])],  1))
        x21 = self.d21(torch.cat([self.ag["21"](f[3], f[2]), self.up(f[3])],  1))
        x31 = self.d31(torch.cat([self.ag["31"](f[4], f[3]), self.up(f[4])],  1))

        x02 = self.d02(torch.cat([f[0], self.ag["02"](x11, x01), self.up(x11)], 1))
        x12 = self.d12(torch.cat([f[1], self.ag["12"](x21, x11), self.up(x21)], 1))
        x22 = self.d22(torch.cat([f[2], self.ag["22"](x31, x21), self.up(x31)], 1))

        x03 = self.d03(torch.cat([f[0], x01, self.ag["03"](x12, x02), self.up(x12)], 1))
        x13 = self.d13(torch.cat([f[1], x11, self.ag["13"](x22, x12), self.up(x22)], 1))
        x04 = self.d04(torch.cat([f[0], x01, x02, self.ag["04"](x13, x03), self.up(x13)], 1))

        H, W = x.shape[2:]
        if self.deep_supervision:
            return [
                F.interpolate(h(o), size=(H, W), mode="bilinear", align_corners=True)
                for h, o in zip(self.heads, [x01, x02, x03, x04])
            ]
        out = self.head(x04)
        return F.interpolate(out, size=(H, W), mode="bilinear", align_corners=True)

# =====================================================================
# 8. EMA
# =====================================================================
class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay  = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.copy_(self.decay * s + (1 - self.decay) * m.data)
        for s, m in zip(self.shadow.buffers(), model.buffers()):
            s.copy_(m)

    def __call__(self, *a, **kw):
        return self.shadow(*a, **kw)

# =====================================================================
# 9. TTA
# =====================================================================
@torch.no_grad()
def tta_predict(model, imgs, device):
    logit_sum = torch.zeros_like(imgs[:, :1, :, :])
    count = 0
    for fh in [False, True]:
        for fv in [False, True]:
            x = imgs.clone()
            if fh: x = torch.flip(x, [3])
            if fv: x = torch.flip(x, [2])
            with torch.amp.autocast("cuda"):
                out = model(x)
                logits = out[-1] if isinstance(out, list) else out
            if fv: logits = torch.flip(logits, [2])
            if fh: logits = torch.flip(logits, [3])
            logit_sum = logit_sum + logits.float()
            count += 1
    return logit_sum / count

# =====================================================================
# 10. SLIDING-WINDOW INFERENCE
# =====================================================================
@torch.no_grad()
def sliding_window_inference(
    model, image_np, device,
    patch_size=INFER_PATCH_SIZE, stride=INFER_STRIDE, use_tta=False,
):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    H, W       = image_np.shape[:2]
    logit_acc  = np.zeros((1, 1, H, W), dtype=np.float32)
    count_acc  = np.zeros((1, 1, H, W), dtype=np.float32)

    pad_h      = (stride - (H - patch_size) % stride) % stride
    pad_w      = (stride - (W - patch_size) % stride) % stride
    img_padded = cv2.copyMakeBorder(image_np, 0, pad_h, 0, pad_w,
                                    cv2.BORDER_CONSTANT, value=(0, 0, 0))
    Hp, Wp = img_padded.shape[:2]

    for y in range(0, Hp - patch_size + 1, stride):
        for x in range(0, Wp - patch_size + 1, stride):
            patch   = img_padded[y:y+patch_size, x:x+patch_size]
            patch_t = (torch.from_numpy(
                ((patch.astype(np.float32) / 255.0 - mean) / std)
                .transpose(2, 0, 1))
                .unsqueeze(0).to(device))

            with torch.amp.autocast("cuda"):
                if use_tta:
                    logits = tta_predict(model, patch_t, device)
                else:
                    out    = model(patch_t)
                    logits = out[-1] if isinstance(out, list) else out

            logits_np = logits.float().cpu().numpy()
            y_end = min(y + patch_size, H); x_end = min(x + patch_size, W)
            ph = y_end - y;                 pw = x_end - x
            logit_acc[:, :, y:y_end, x:x_end] += logits_np[:, :, :ph, :pw]
            count_acc[:, :, y:y_end, x:x_end] += 1.0

    count_acc = np.maximum(count_acc, 1.0)
    return (logit_acc / count_acc)[0, 0]

# =====================================================================
# 11. ★ COMPUTATIONAL COMPLEXITY ANALYSIS ★
# =====================================================================
def compute_complexity(model, device, input_size=(1, 3, 512, 512)):
    """
    Returns a dict with:
        params_M   : trainable parameters in millions
        gflops     : GFLOPs (via torchinfo or ptflops; None if unavailable)
        fps        : frames per second (GPU timed, batch=1)
        latency_ms : mean single-image inference latency in ms
    """
    model.eval()

    # ── 1. Param count ───────────────────────────────────────────────
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    params_M = params / 1e6

    # ── 2. FLOPs ─────────────────────────────────────────────────────
    gflops = None
    if HAS_TORCHINFO:
        try:
            stats  = torchinfo_summary(model, input_size=input_size,
                                       verbose=0, device=str(device))
            gflops = stats.total_mult_adds / 1e9
        except Exception as e:
            print(f"  torchinfo FLOPs failed: {e}")

    if gflops is None and HAS_PTFLOPS:
        try:
            macs, _ = get_model_complexity_info(
                model, input_size[1:], as_strings=False,
                print_per_layer_stat=False, verbose=False)
            gflops = macs / 1e9
        except Exception as e:
            print(f"  ptflops FLOPs failed: {e}")

    if gflops is None:
        print("  ⚠ FLOPs unavailable (install torchinfo: pip install torchinfo)")

    # ── 3. FPS / latency ─────────────────────────────────────────────
    dummy = torch.randn(input_size, device=device)
    N_warm, N_time = 5, 30

    with torch.no_grad():
        for _ in range(N_warm):
            _ = model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N_time):
            _ = model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed    = time.perf_counter() - t0
    fps        = N_time / elapsed
    latency_ms = (elapsed / N_time) * 1000.0

    print("\n" + "═" * 55)
    print(f"  COMPLEXITY ANALYSIS  [{ABL_TAG}]")
    print("═" * 55)
    print(f"  Trainable params  : {params_M:>8.2f} M  ({params:,})")
    if gflops is not None:
        print(f"  GFLOPs (512×512)  : {gflops:>8.2f}")
    else:
        print(f"  GFLOPs (512×512)  :     N/A")
    print(f"  FPS  (batch=1)    : {fps:>8.1f}")
    print(f"  Latency (ms/img)  : {latency_ms:>8.1f}")
    print("═" * 55 + "\n")

    return dict(params_M=round(params_M, 2),
                gflops=round(gflops, 2) if gflops else None,
                fps=round(fps, 1),
                latency_ms=round(latency_ms, 2))

# =====================================================================
# 12. ★ METRIC SAVING  ★
# =====================================================================
METRIC_CSV  = f"metrics_history_{ABL_TAG}.csv"
SUMMARY_JSON = f"run_summary_{ABL_TAG}.json"

_csv_initialized = False

def save_metrics_csv(epoch, train_loss, vd, vi, vp, vr, vf1, lr):
    global _csv_initialized
    fieldnames = ["epoch", "train_loss", "dice", "iou",
                  "precision", "recall", "f1", "lr"]
    write_header = not _csv_initialized and not os.path.exists(METRIC_CSV)
    with open(METRIC_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(dict(
            epoch=epoch, train_loss=round(train_loss, 6),
            dice=round(vd, 6), iou=round(vi, 6),
            precision=round(vp, 6), recall=round(vr, 6),
            f1=round(vf1, 6), lr=f"{lr:.2e}",
        ))
    _csv_initialized = True


def save_summary_json(best_dice, best_iou, best_f1,
                      best_epoch, complexity_info, epochs_run):
    data = dict(
        ablation      = ABLATION,
        tag           = ABL_TAG,
        epochs_run    = epochs_run,
        best_epoch    = best_epoch,
        best_dice     = round(best_dice, 6),
        best_iou      = round(best_iou, 6),
        best_f1       = round(best_f1, 6),
        complexity    = complexity_info,
    )
    with open(SUMMARY_JSON, "w") as f:
        json.dump(data, f, indent=2)
    print(f"  ✓ Summary saved → {SUMMARY_JSON}")

# =====================================================================
# 13. PLOTTING
# =====================================================================
def plot_all(history, path=None):
    if path is None:
        path = f"training_metrics_{ABL_TAG}.png"
    eps = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Mamba-UNet++ v4 DFUC2022 [{ABL_TAG}]",
                 fontsize=16, fontweight="bold")

    def _p(a, key, label, color, target=None):
        a.plot(eps, history[key], color=color, marker="o", markersize=2, label=label)
        if target:
            a.axhline(target, color="limegreen", ls="--", lw=1,
                      alpha=0.7, label=f"Target {target}")
        a.legend(fontsize=8); a.grid(True, alpha=0.3)

    _p(ax[0,0], "train_loss", "Train Loss",  "red")
    ax[0,0].set_title("Loss")
    _p(ax[0,1], "val_dice",   "Val Dice",    "royalblue", 0.73)
    ax[0,1].set_title("Dice"); ax[0,1].set_ylim(0, 1)
    _p(ax[0,2], "val_iou",    "Val IoU",     "seagreen")
    ax[0,2].set_title("IoU"); ax[0,2].set_ylim(0, 1)

    ax[1,0].plot(eps, history["val_precision"], color="purple",
                 marker="o", markersize=2, label="Precision")
    ax[1,0].plot(eps, history["val_recall"],    color="teal",
                 marker="s", markersize=2, label="Recall")
    ax[1,0].plot(eps, history["val_f1"],        color="orange",
                 marker="^", markersize=2, label="F1")
    ax[1,0].legend(fontsize=8); ax[1,0].grid(True, alpha=0.3)
    ax[1,0].set_ylim(0, 1); ax[1,0].set_title("Precision / Recall / F1")

    ax[1,1].axis("off")
    ax[1,1].text(0.5, 0.5,
                 f"Ablation: {ABLATION}\n\nHD95 computed\nonly at inference",
                 ha="center", va="center", fontsize=12,
                 style="italic", color="gray")

    ax[1,2].plot(eps, history["lr"], "k", marker="o", markersize=2)
    ax[1,2].set_title("Learning Rate")
    ax[1,2].set_yscale("log"); ax[1,2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Plot saved → {path}")

# =====================================================================
# 14. INFERENCE VISUALISATION
# =====================================================================
def run_inference(model_path, image_dir, mask_dir, num_samples=4, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # load with matching ablation settings
    deep_sup = (ABLATION != "no_deep_sup")
    model = MambaUNetPPv4(
        deep_supervision=deep_sup, pretrained=False, ablation=ABLATION
    ).to(device)
    sd = torch.load(model_path, map_location=device)
    model.load_state_dict(sd, strict=False)
    model.eval()
    print(f"\n✓ Loaded checkpoint from '{model_path}'")

    ext  = (".png", ".jpg", ".jpeg")
    imgs = sorted([f for f in os.listdir(image_dir)
                   if f.lower().endswith(ext)])
    picks = random.sample(imgs, min(num_samples, len(imgs)))

    fig, axes = plt.subplots(num_samples, 4, figsize=(17, 4*num_samples))
    if num_samples == 1:
        axes = axes[np.newaxis]
    fig.suptitle(
        f"Inference — Mamba-UNet++ v4 DFUC2022 [{ABL_TAG}]",
        fontsize=14, fontweight="bold")
    for c, t in enumerate(["Input", "Ground Truth",
                            "Prediction", "Overlay (TP/FP/FN)"]):
        axes[0, c].set_title(t, fontsize=11, fontweight="bold")

    all_hd95 = []
    for row, name in enumerate(picks):
        ip = os.path.join(image_dir, name)
        mp = os.path.join(mask_dir, name.rsplit(".", 1)[0] + ".png")
        if not os.path.exists(mp):
            mp = os.path.join(mask_dir, name)

        orig    = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        orig_h, orig_w = orig.shape[:2]
        gt_raw  = (cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
                   if os.path.exists(mp)
                   else np.zeros((orig_h, orig_w), np.uint8))
        gt      = (gt_raw > 127).astype(np.float32)

        orig_sq, gt_sq, pad_info = pad_to_square(orig, gt, pad_value=0)
        sq_h, sq_w = orig_sq.shape[:2]

        logit_map = sliding_window_inference(
            model, orig_sq, device,
            patch_size=INFER_PATCH_SIZE, stride=INFER_STRIDE, use_tta=True)
        pred_sq = (1 / (1 + np.exp(-logit_map)) > 0.5).astype(np.float32)
        pred    = unpad(pred_sq, pad_info, orig_h, orig_w)
        pred    = (pred > 0.5).astype(np.float32)

        gtt   = torch.from_numpy(gt).unsqueeze(0).unsqueeze(0)
        pt    = torch.from_numpy(pred).unsqueeze(0).unsqueeze(0)
        dice  = float((2*(pt*gtt).sum()+1e-6) / (pt.sum()+gtt.sum()+1e-6))
        iou   = float(((pt*gtt).sum()+1e-6) / ((pt+gtt-(pt*gtt)).sum()+1e-6))
        prec  = float((pt*gtt).sum()+1e-6) / float(pt.sum()+1e-6)
        rec   = float((pt*gtt).sum()+1e-6) / float(gtt.sum()+1e-6)
        f1    = 2*prec*rec / (prec + rec + 1e-6)

        logit_t = torch.from_numpy(logit_map).unsqueeze(0).unsqueeze(0)
        gt_rsz  = cv2.resize(gt, (sq_w, sq_h), interpolation=cv2.INTER_NEAREST)
        gt_t    = torch.from_numpy(gt_rsz).unsqueeze(0).unsqueeze(0)
        hd      = hausdorff_95(logit_t, gt_t)
        all_hd95.append(hd)

        disp_size = (512, 512)
        orig_disp = cv2.resize(orig, disp_size)
        gt_disp   = cv2.resize(gt,   disp_size, interpolation=cv2.INTER_NEAREST)
        pred_disp = cv2.resize(pred, disp_size, interpolation=cv2.INTER_NEAREST)

        ov = orig_disp.copy().astype(np.float32)
        tp_m = (pred_disp > 0.5) & (gt_disp > 0.5)
        fp_m = (pred_disp > 0.5) & (gt_disp <= 0.5)
        fn_m = (pred_disp <= 0.5) & (gt_disp > 0.5)
        ov[tp_m] = ov[tp_m]*0.45 + np.array([0, 220, 0])*0.55
        ov[fp_m] = ov[fp_m]*0.45 + np.array([220, 0, 0])*0.55
        ov[fn_m] = ov[fn_m]*0.45 + np.array([0, 0, 220])*0.55

        axes[row,0].imshow(orig_disp)
        axes[row,0].set_ylabel(name[:18], fontsize=7)
        axes[row,1].imshow(gt_disp, cmap="gray")
        axes[row,2].imshow(pred_disp, cmap="gray")
        axes[row,2].set_xlabel(
            f"Dice={dice:.4f}  IoU={iou:.4f}  F1={f1:.4f}  HD95={hd:.1f}px",
            fontsize=8)
        axes[row,3].imshow(np.clip(ov, 0, 255).astype(np.uint8))
        for a in axes[row]:
            a.axis("off")

    patches = [
        mpatches.Patch(color="lime",  label="TP"),
        mpatches.Patch(color="red",   label="FP"),
        mpatches.Patch(color="blue",  label="FN"),
    ]
    fig.legend(handles=patches, loc="lower center", ncol=3,
               fontsize=10, bbox_to_anchor=(0.5, -0.005))
    plt.tight_layout()
    out_path = f"inference_results_{ABL_TAG}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"✓ Inference grid saved → '{out_path}'")
    print(f"  Mean HD95 across samples: {np.mean(all_hd95):.1f} px")

# =====================================================================
# 15. TRAINING
# =====================================================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice : {device}")
    if torch.cuda.is_available():
        print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Ablation : {ABLATION}  |  tag: {ABL_TAG}")
    print(f"Epochs   : {EPOCHS}   |  Patience: {PATIENCE}\n")

    # ── dataset paths ─────────────────────────────────────────────────
    base    = "dataset/train"
    img_dir = os.path.join(base, "images")
    msk_dir = os.path.join(base, "labels")
    if not os.path.exists(img_dir):
        img_dir = msk_dir = base

    ext  = (".png", ".jpg", ".jpeg")
    all_ = sorted([f for f in os.listdir(img_dir)
                   if f.lower().endswith(ext)])
    if not all_:
        raise ValueError(f"No images in {img_dir}")

    random.shuffle(all_)
    split    = int(0.8 * len(all_))
    tr_names = all_[:split];  vl_names = all_[split:]
    print(f"Train: {len(tr_names)} | Val: {len(vl_names)}")
    print(f"Strategy: {TRAIN_PATCH_SIZE}×{TRAIN_PATCH_SIZE} crops (train) | "
          f"pad-to-square {PAD_RESIZE}px (val)\n")

    # ── datasets ──────────────────────────────────────────────────────
    tr_ds = DFUC2022Dataset(img_dir, msk_dir, tr_names,
                             transform=get_train_transforms(),
                             training=True, patch_size=TRAIN_PATCH_SIZE,
                             crops_per_image=2)
    vl_ds = DFUC2022Dataset(img_dir, msk_dir, vl_names,
                             transform=get_val_transforms(), training=False)

    BATCH = 8
    tr_ld = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,
                       num_workers=4, pin_memory=True, persistent_workers=True)
    vl_ld = DataLoader(vl_ds, batch_size=4, shuffle=False,
                       num_workers=4, pin_memory=True, persistent_workers=True)

    # ── model ─────────────────────────────────────────────────────────
    deep_sup = (ABLATION != "no_deep_sup")
    model    = MambaUNetPPv4(
        deep_supervision=deep_sup, pretrained=HAS_TIMM,
        ablation=ABLATION,
    ).to(device)

    print("⏭  Skipping torch.compile (avoids pthread exhaustion)")

    # ── complexity report (before training) ───────────────────────────
    complexity_info = compute_complexity(model, device)

    # ── EMA + loss ────────────────────────────────────────────────────
    ema  = EMA(model, decay=0.9995)
    crit = ScheduledLoss(dice_bce_only=(ABLATION == "dice_bce_only"))

    # ── optimiser — differential LR ───────────────────────────────────
    enc_params    = list(model.encoder.parameters())
    enc_param_ids = {id(p) for p in enc_params}
    dec_params    = [p for p in model.parameters()
                     if id(p) not in enc_param_ids]

    optimizer = torch.optim.AdamW(
        [{"params": enc_params, "lr": 3e-5, "name": "encoder"},
         {"params": dec_params, "lr": 2e-4, "name": "decoder"}],
        weight_decay=1e-4, betas=(0.9, 0.999),
    )
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[3e-4, 2e-3],
        epochs=EPOCHS, steps_per_epoch=len(tr_ld),
        pct_start=0.05, anneal_strategy="cos",
        div_factor=10, final_div_factor=100,
    )
    scaler = torch.amp.GradScaler("cuda")

    ds_w  = [0.05, 0.15, 0.30, 0.50]
    ACCUM = 2

    best_dice  = 0.0;  best_iou  = 0.0;  best_f1   = 0.0
    best_epoch = 0;    patience  = 0
    history = {k: [] for k in ["train_loss", "val_dice", "val_iou",
                                "val_precision", "val_recall", "val_f1", "lr"]}

    ckpt_name = f"best_mamba_unetpp_v4_{ABL_TAG}.pth"
    print(f"Training: {EPOCHS} epochs | OneCycleLR | AdamW | EMA\n")

    for epoch in range(EPOCHS):
        model.train()
        crit.set_epoch(epoch)
        tloss = 0.0;  n_valid = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(enumerate(tr_ld), total=len(tr_ld),
                    desc=f"E{epoch+1:03d}/{EPOCHS} [Train]", ncols=110)

        for step, (imgs, masks) in pbar:
            imgs  = imgs.to(device,  non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.amp.autocast("cuda"):
                seg_outs = model(imgs)
                if isinstance(seg_outs, tuple): seg_outs = seg_outs[0]
                if not isinstance(seg_outs, list): seg_outs = [seg_outs]
                loss = (sum(w * crit(o, masks)
                            for w, o in zip(ds_w[-len(seg_outs):], seg_outs))
                        / ACCUM)

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                pbar.set_postfix({"loss": "nan-skip"})
                continue

            scaler.scale(loss).backward()
            n_valid += 1

            if (step + 1) % ACCUM == 0 or (step + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)

            sched.step()
            loss_val = loss.item() * ACCUM
            tloss   += loss_val
            pbar.set_postfix({"loss": f"{loss_val:.4f}"})

        epoch_loss = tloss / max(n_valid, 1)

        # ── validation ────────────────────────────────────────────────
        ema.shadow.eval()
        d_l, i_l, p_l, r_l, f1_l = [], [], [], [], []
        use_tta = (epoch + 1) >= 80 and (epoch + 1) % 5 == 0

        with torch.no_grad():
            for imgs, masks in tqdm(vl_ld,
                                    desc=f"E{epoch+1:03d}/{EPOCHS} [Val]  ",
                                    leave=False, ncols=110):
                imgs  = imgs.to(device,  non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                if use_tta:
                    final_out = tta_predict(ema.shadow, imgs, device)
                else:
                    with torch.amp.autocast("cuda"):
                        out = ema.shadow(imgs)
                        if isinstance(out, tuple): out = out[0]
                        final_out = out[-1] if isinstance(out, list) else out

                d, i, p, r, f1 = calc_metrics(final_out, masks)
                d_l.append(d);  i_l.append(i);  p_l.append(p)
                r_l.append(r);  f1_l.append(f1)

        vd, vi, vp, vr, vf1 = (np.mean(d_l), np.mean(i_l), np.mean(p_l),
                                np.mean(r_l), np.mean(f1_l))
        cur_lr = optimizer.param_groups[0]["lr"]

        # ── log ───────────────────────────────────────────────────────
        history["train_loss"].append(epoch_loss)
        history["val_dice"].append(vd);      history["val_iou"].append(vi)
        history["val_precision"].append(vp); history["val_recall"].append(vr)
        history["val_f1"].append(vf1);       history["lr"].append(cur_lr)

        # ── auto-save CSV ─────────────────────────────────────────────
        save_metrics_csv(epoch + 1, epoch_loss, vd, vi, vp, vr, vf1, cur_lr)

        tta_tag = " [TTA]" if use_tta else ""
        print(f"  Loss:{epoch_loss:.4f} | Dice:{vd:.4f} | IoU:{vi:.4f} | "
              f"Prec:{vp:.4f} | Rec:{vr:.4f} | F1:{vf1:.4f} | "
              f"LR:{cur_lr:.1e}{tta_tag}")

        # ── checkpoint ────────────────────────────────────────────────
        if vd > best_dice:
            best_dice  = vd;  best_iou = vi;  best_f1 = vf1
            best_epoch = epoch + 1
            torch.save(ema.shadow.state_dict(), ckpt_name)
            print(f"  ✓ Best model saved → {ckpt_name}  (Dice={best_dice:.4f})")
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f"\nEarly stop at epoch {epoch+1}. "
                      f"Best Dice={best_dice:.4f}")
                break

        if (epoch + 1) % 5 == 0:
            plot_all(history)

    # ── final summary ─────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f" Training done | Best EMA Dice: {best_dice:.4f}  "
          f"IoU: {best_iou:.4f}  F1: {best_f1:.4f}")
    print(f" Best epoch    : {best_epoch}")
    print(f"{'='*65}")

    plot_all(history)
    save_summary_json(best_dice, best_iou, best_f1,
                      best_epoch, complexity_info, len(history["train_loss"]))
    print(f"  ✓ Metrics history → {METRIC_CSV}")

    run_inference(ckpt_name, img_dir, msk_dir, num_samples=4, device=device)


if __name__ == "__main__":

    # ── ABLATION QUICK-REFERENCE ─────────────────────────────────────
    # Change ABLATION at top of file, then re-run.
    # Each run saves to separate files with its own ABL_TAG prefix:
    #
    #   ABLATION = "none"             → full model  (full)
    #   ABLATION = "no_mamba"         → Mamba → Identity
    #   ABLATION = "no_deep_sup"      → single output head
    #   ABLATION = "no_bidirectional" → forward scan only
    #   ABLATION = "dice_bce_only"    → no Tversky/Lovász
    # ─────────────────────────────────────────────────────────────────

    train()

✓ Fast CUDA Mamba loaded
✓ timm loaded — will use pretrained EfficientNet-B4 encoder

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB
Ablation : none  |  tag: full
Epochs   : 50   |  Patience: 50

Train: 1656 | Val: 415
Strategy: 512×512 crops (train) | pad-to-square 640px (val)



Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  EfficientNet-B4 encoder channels: [24, 32, 56, 160, 448]
⏭  Skipping torch.compile (avoids pthread exhaustion)

═══════════════════════════════════════════════════════
  COMPLEXITY ANALYSIS  [full]
═══════════════════════════════════════════════════════
  Trainable params  :    28.02 M  (28,021,728)
  GFLOPs (512×512)  :   119.53
  FPS  (batch=1)    :     39.1
  Latency (ms/img)  :     25.6
═══════════════════════════════════════════════════════

Training: 50 epochs | OneCycleLR | AdamW | EMA



E001/50 [Train]: 100%|█████████████████████████████████████████| 414/414 [02:04<00:00,  3.32it/s, loss=0.2396]
                                                                                                              

  Loss:0.9008 | Dice:0.0209 | IoU:0.0113 | Prec:0.0774 | Rec:0.1242 | F1:0.2878 | LR:1.2e-04
  ✓ Best model saved → best_mamba_unetpp_v4_full.pth  (Dice=0.0209)


E002/50 [Train]: 100%|█████████████████████████████████████████| 414/414 [01:56<00:00,  3.54it/s, loss=0.4907]
                                                                                                              

... (11 outputs left out)

E008/50 [Train]: 100%|█████████████████████████████████████████| 414/414 [01:57<00:00,  3.52it/s, loss=0.1715]
                                                                                                              

  Loss:0.2141 | Dice:0.1449 | IoU:0.1262 | Prec:0.9902 | Rec:0.1286 | F1:0.1514 | LR:2.9e-04
  ✓ Best model saved → best_mamba_unetpp_v4_full.pth  (Dice=0.1449)


E009/50 [Train]:  23%|█████████▋                                | 95/414 [00:27<01:32,  3.45it/s, loss=0.1297]


KeyboardInterrupt: 

In [4]:
# """
# Mamba-UNet++ v4 — DFUC2022 Edition (Targeting ~0.73+ SOTA Dice)
# ================================================================
# With Auto-Metrics, Complexity Analysis, and Zero-Collapse Fixes
# """

# ── ABLATION & TRAINING CONFIGURATION ────────────────────────────────
CONFIG = {
    "epochs": 50,
    "patience": 20,              # Less aggressive early stopping
    "use_mamba": True,           # False = replace Mamba with Identity
    "use_deep_supervision": True,# False = final output only
    "bidirectional_mamba": True, # False = disable reverse scan
    "use_loss_schedule": True    # False = Dice/BCE only
}
# ─────────────────────────────────────────────────────────────────────

import os

# Thread guards MUST come before cv2 / numpy import
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import math, random, warnings, csv, time
import cv2

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt
from copy import deepcopy

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── optional thop for FLOPs ──────────────────────────────────────────
try:
    from thop import profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("⚠  thop not found. Install via `pip install thop` for exact FLOPs.")

# ── optional fast Mamba ──────────────────────────────────────────────
try:
    from mamba_ssm import Mamba
    USE_FAST_MAMBA = True
    print("✓ Fast CUDA Mamba loaded")
except ImportError:
    USE_FAST_MAMBA = False
    print("⚠  mamba-ssm not found — using CBAM attention (fast fallback)")

# ── optional timm ────────────────────────────────────────────────────
try:
    import timm
    HAS_TIMM = True
    print("✓ timm loaded — will use pretrained EfficientNet-B4 encoder")
except ImportError:
    HAS_TIMM = False
    print("⚠  timm not found — falling back to scratch encoder")

# ── DFUC2022 constants ───────────────────────────────────────────────
TRAIN_PATCH_SIZE = 512
INFER_PATCH_SIZE = 512
INFER_STRIDE = 384
PAD_RESIZE = 640
IMG_SIZE = TRAIN_PATCH_SIZE
image_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_images'
mask_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_masks'

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()

# =====================================================================
# 1. COMPLEXITY ANALYSIS
# =====================================================================
def analyze_complexity(model, device, input_size=(1, 3, 512, 512)):
    print("\n[Profiling Hardware Complexity...]")
    dummy_input = torch.randn(input_size).to(device)
    model.eval()
    
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    flops = 0
    if HAS_THOP:
        with torch.no_grad():
            macs, _ = profile(deepcopy(model).to(device), inputs=(dummy_input,), verbose=False)
            flops = macs * 2
    
    runs = 100
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for _ in range(10): model(dummy_input)
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(runs): model(dummy_input)
        torch.cuda.synchronize()
        end_time = time.time()
        
    total_time = end_time - start_time
    time_per_img_ms = (total_time / runs) * 1000
    fps = 1000.0 / time_per_img_ms
    
    print(f"  ➜ Params: {params / 1e6:.2f} M")
    if HAS_THOP: print(f"  ➜ FLOPs : {flops / 1e9:.2f} G")
    print(f"  ➜ FPS   : {fps:.1f} imgs/sec ({time_per_img_ms:.2f} ms/img)\n")
    return params, flops, fps, time_per_img_ms

# =====================================================================
# 2. HELPERS
# =====================================================================
def pad_to_square(image, mask=None, pad_value=0):
    h, w = image.shape[:2]
    if h == w:
        if mask is not None: return image, mask, (0, 0, 0, 0)
        return image, (0, 0, 0, 0)
    diff = abs(h - w)
    pad_before = diff // 2
    pad_after = diff - pad_before
    top, bottom, left, right = (0, 0, pad_before, pad_after) if h > w else (pad_before, pad_after, 0, 0)
    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(pad_value, pad_value, pad_value))
    if mask is not None:
        mask = cv2.copyMakeBorder(mask, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)
        return image, mask, (top, bottom, left, right)
    return image, (top, bottom, left, right)

# =====================================================================
# 3. DATASET
# =====================================================================
class DFUC2022Dataset(Dataset):
    def __init__(self, image_dir, mask_dir, names, transform=None, training=True, patch_size=TRAIN_PATCH_SIZE, crops_per_image=1):
        self.transform = transform
        self.training = training
        self.patch_size = patch_size
        self.crops_per_image = crops_per_image
        self.samples = []
        for n in names:
            ip = os.path.join(image_dir, n)
            if not os.path.exists(ip) or os.path.getsize(ip) == 0: continue
            stem = n.rsplit(".", 1)[0]
            mp_png = os.path.join(mask_dir, stem + ".png")
            mp_jpg = os.path.join(mask_dir, n)
            mp = mp_png if os.path.exists(mp_png) else (mp_jpg if os.path.exists(mp_jpg) else None)
            self.samples.append({"img": ip, "mask": mp})

    def __len__(self):
        return len(self.samples) * self.crops_per_image if self.training else len(self.samples)

    @staticmethod
    def _bbox_of_mask(mask):
        ys, xs = np.where(mask > 0.5)
        if len(ys) == 0: return None
        return int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())

    def _random_crop_around_fg(self, image, mask):
        ps = self.patch_size
        h, w = image.shape[:2]
        pad_h, pad_w = max(ps - h, 0), max(ps - w, 0)
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=(0, 0, 0))
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            h, w = image.shape[:2]

        bbox = self._bbox_of_mask(mask)
        if bbox is not None:
            y0, y1, x0, x1 = bbox
            cy_lo, cy_hi = max(0, y1 - ps + 1), min(h - ps, y0)
            cx_lo, cx_hi = max(0, x1 - ps + 1), min(w - ps, x0)
            cy = random.randint(min(cy_lo, cy_hi), max(cy_lo, cy_hi))
            cx = random.randint(min(cx_lo, cx_hi), max(cx_lo, cx_hi))
        else:
            cy, cx = random.randint(0, h - ps), random.randint(0, w - ps)
        return image[cy : cy + ps, cx : cx + ps], mask[cy : cy + ps, cx : cx + ps]

    def __getitem__(self, idx):
        s = self.samples[idx % len(self.samples)]
        im = cv2.imread(s["img"])
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im is not None else np.zeros((TRAIN_PATCH_SIZE, TRAIN_PATCH_SIZE, 3), np.uint8)
        
        # FIXED: Absolute > 0 catch-all to prevent black-mask collapse
        mk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE) if s["mask"] else np.zeros(im.shape[:2], np.uint8)
        mk = (mk > 0).astype(np.float32)

        if self.training:
            im, mk = self._random_crop_around_fg(im, mk)
        else:
            im, mk, _ = pad_to_square(im, mk, pad_value=0)
            im = cv2.resize(im, (PAD_RESIZE, PAD_RESIZE))
            mk = cv2.resize(mk, (PAD_RESIZE, PAD_RESIZE), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            aug = self.transform(image=im, mask=mk)
            im, mk = aug["image"], aug["mask"].unsqueeze(0)
        return im, mk

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-30, 30), p=0.5),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05, p=0.2), A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.4), A.HueSaturationValue(8, 20, 10, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

# =====================================================================
# 4. METRICS
# =====================================================================
def calc_metrics(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float().cpu().numpy()
    tgt = target.float().cpu().numpy()
    dice_l, iou_l, prec_l, rec_l, f1_l = [], [], [], [], []
    for p, t in zip(pred, tgt):
        p, t = p.squeeze(), t.squeeze()
        tp = float((p * t).sum())
        fp = float((p * (1 - t)).sum())
        fn = float(((1 - p) * t).sum())
        dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
        dice_l.append(dice)
        iou_l.append((tp + 1e-6) / (tp + fp + fn + 1e-6))
        prec_l.append((tp + 1e-6) / (tp + fp + 1e-6))
        rec_l.append((tp + 1e-6) / (tp + fn + 1e-6))
        f1_l.append(dice)
    return map(np.mean, [dice_l, iou_l, prec_l, rec_l, f1_l])

# =====================================================================
# 5. LOSSES
# =====================================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce = self.bce(logits.float(), targets.float())
        p = torch.sigmoid(logits.float()).view(-1)
        t = targets.float().view(-1)
        dice = 1 - (2 * (p * t).sum() + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce, dice

class LovaszHingeLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits.float()).view(-1)
        tgts = targets.float().view(-1)
        if tgts.sum() == 0 or tgts.sum() == tgts.numel(): return (logits * 0.0).mean()
        errors = (tgts - probs).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        gts = tgts[perm].sum()
        intersection = gts - tgts[perm].cumsum(0)
        union = gts + (1 - tgts[perm]).cumsum(0)
        jaccard = 1.0 - intersection / (union + 1e-6)
        if tgts[perm].numel() > 1: jaccard[1:] = jaccard[1:] - jaccard[:-1]
        return torch.clamp(torch.dot(errors_sorted, jaccard), 0.0, 2.0)

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-3):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs.float()).view(-1)
        t = targets.float().view(-1)
        tp = (p * t).sum()
        fp = (p * (1 - t)).sum()
        fn = ((1 - p) * t).sum()
        tv = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        tv = torch.clamp(tv, min=1e-6, max=1.0)
        return (1 - tv) ** self.gamma

class ScheduledLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice_bce = DiceBCELoss()
        self.tversky = TverskyLoss()
        self.lovasz = LovaszHingeLoss()
        self._epoch = 0

    def set_epoch(self, e): self._epoch = e

    def forward(self, logits, targets):
        bce, dice = self.dice_bce(logits, targets)
        
        if not CONFIG["use_loss_schedule"]:
            return bce + dice
            
        # FIXED: Warm-up period. Pure BCE/Dice for 5 epochs to prevent Lovasz/Tversky NaN explosion
        if self._epoch < 5: 
            return bce + dice
            
        tv = self.tversky(logits, targets)
        lov = self.lovasz(logits, targets)
        def safe(t): return t if torch.isfinite(t) else torch.zeros(1, device=logits.device)[0]
        
        e = self._epoch
        if e < 20:   w = (0.25, 0.25, 0.40, 0.10)
        elif e < 50: w = (0.20, 0.30, 0.30, 0.20)
        else:        w = (0.15, 0.30, 0.25, 0.30)
        return w[0] * safe(bce) + w[1] * safe(dice) + w[2] * safe(tv) + w[3] * safe(lov)

# =====================================================================
# 6. BUILDING BLOCKS
# =====================================================================
class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.se(x).view(x.size(0), x.size(1), 1, 1)

class CBAM(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False))
        self.spatial = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, x):
        ca = torch.sigmoid(self.mlp(x.mean(dim=(2, 3))) + self.mlp(x.amax(dim=(2, 3)))).unsqueeze(-1).unsqueeze(-1)
        x = x * ca
        sa = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * self.spatial(sa)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, drop=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU(), nn.Dropout2d(drop),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.se = SEBlock(out_c)
        self.proj = nn.Conv2d(in_c, out_c, 1, bias=False) if in_c != out_c else nn.Identity()
    def forward(self, x): return self.se(self.conv(x)) + self.proj(x)

class ASPP(nn.Module):
    def __init__(self, in_c, out_c=256):
        super().__init__()
        rates = [1, 6, 12, 18]
        self.convs = nn.ModuleList([nn.Sequential(nn.Conv2d(in_c, out_c, 3 if r>1 else 1, padding=r if r>1 else 0, dilation=r, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)) for r in rates])
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_c, out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
        self.proj = nn.Sequential(nn.Conv2d(out_c * (len(rates) + 1), out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
    def forward(self, x):
        h, w = x.shape[2:]
        feats = [c(x) for c in self.convs] + [F.interpolate(self.pool(x), size=(h, w), mode="bilinear", align_corners=True)]
        return self.proj(torch.cat(feats, 1))

class AttentionGate(nn.Module):
    def __init__(self, Fg, Fl, Fi):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(Fg, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.Wx = nn.Sequential(nn.Conv2d(Fl, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        g1 = self.Wg(F.interpolate(g, x.shape[2:], mode="bilinear", align_corners=True))
        return x * self.psi(F.relu(g1 + self.Wx(x), inplace=True))

class FastMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mamba = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.norm = nn.LayerNorm(d)

    def _scan(self, x, flip=False):
        s = torch.flip(x, [1]) if flip else x
        o = self.mamba(self.norm(s))
        return torch.flip(o, [1]) if flip else o

    def forward(self, x):
        # Cast to float32 internally to prevent Mamba hidden state fp16 overflow
        with torch.amp.autocast("cuda", enabled=False):
            x_f32 = x.float()
            B, C, H, W = x_f32.shape
            h = x_f32.view(B, C, -1).transpose(1, 2)
            v = x_f32.permute(0, 1, 3, 2).contiguous().view(B, C, -1).transpose(1, 2)
            
            if CONFIG["bidirectional_mamba"]:
                oh = (self._scan(h) + self._scan(h, flip=True)) / 2
                ov = (self._scan(v) + self._scan(v, flip=True)) / 2
            else:
                oh = self._scan(h)
                ov = self._scan(v)
                
            out = ((oh + ov) / 2).transpose(1, 2).view(B, C, H, W) + x_f32
            return out.to(x.dtype)

# =====================================================================
# 7. ENCODER & UNET++
# =====================================================================
class ResidualAdapter(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.needs_proj = in_c != out_c
        if self.needs_proj:
            self.proj = nn.Conv2d(in_c, out_c, 1, bias=False)
            self.bn = nn.BatchNorm2d(out_c)
    def forward(self, x): return self.bn(self.proj(x)) if self.needs_proj else x

class EfficientNetEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.stages = timm.create_model("efficientnet_b4", pretrained=pretrained, features_only=True)
        self.channels = self.stages.feature_info.channels()
    def forward(self, x): return self.stages(x)

class MambaUNetPPv4(nn.Module):
    def __init__(self, num_classes=1, deep_supervision=True, pretrained=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.encoder = EfficientNetEncoder(pretrained=pretrained) if HAS_TIMM else ScratchEncoder()
        
        enc_ch = self.encoder.channels
        dec_ch = [64, 128, 192, 256, 256]
        self.adapt = nn.ModuleList([ResidualAdapter(ec, dc) for ec, dc in zip(enc_ch, dec_ch)])
        self.aspp = ASPP(dec_ch[4], dec_ch[4])
        
        if CONFIG["use_mamba"]:
            self.ctx3 = FastMambaBlock(dec_ch[3]) if USE_FAST_MAMBA else CBAM(dec_ch[3])
            self.ctx4 = FastMambaBlock(dec_ch[4]) if USE_FAST_MAMBA else CBAM(dec_ch[4])
        else:
            self.ctx3 = nn.Identity()
            self.ctx4 = nn.Identity()

        C = dec_ch
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag = nn.ModuleDict({
            "01": AttentionGate(C[1], C[0], C[0]//2), "11": AttentionGate(C[2], C[1], C[1]//2), "21": AttentionGate(C[3], C[2], C[2]//2), "31": AttentionGate(C[4], C[3], C[3]//2),
            "02": AttentionGate(C[1], C[0], C[0]//2), "12": AttentionGate(C[2], C[1], C[1]//2), "22": AttentionGate(C[3], C[2], C[2]//2),
            "03": AttentionGate(C[1], C[0], C[0]//2), "13": AttentionGate(C[2], C[1], C[1]//2), "04": AttentionGate(C[1], C[0], C[0]//2),
        })

        self.d01 = ConvBlock(C[0]+C[1], C[0])
        self.d11 = ConvBlock(C[1]+C[2], C[1])
        self.d21 = ConvBlock(C[2]+C[3], C[2])
        self.d31 = ConvBlock(C[3]+C[4], C[3])
        self.d02 = ConvBlock(C[0]*2+C[1], C[0])
        self.d12 = ConvBlock(C[1]*2+C[2], C[1])
        self.d22 = ConvBlock(C[2]*2+C[3], C[2])
        self.d03 = ConvBlock(C[0]*3+C[1], C[0])
        self.d13 = ConvBlock(C[1]*3+C[2], C[1])
        self.d04 = ConvBlock(C[0]*4+C[1], C[0])

        if deep_supervision: self.heads = nn.ModuleList([nn.Conv2d(C[0], num_classes, 1) for _ in range(4)])
        else: self.head = nn.Conv2d(C[0], num_classes, 1)
        
        # FIXED: Initialize decoder weights to prevent instant signal death
        self._init_weights()

    def _init_weights(self):
        for name, m in self.named_modules():
            if "encoder" not in name:
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        feats = self.encoder(x)
        f = [ad(fe) for ad, fe in zip(self.adapt, feats)]
        f[3] = self.ctx3(f[3])
        f[4] = self.ctx4(self.aspp(f[4]))

        x01 = self.d01(torch.cat([self.ag["01"](f[1], f[0]), self.up(f[1])], 1))
        x11 = self.d11(torch.cat([self.ag["11"](f[2], f[1]), self.up(f[2])], 1))
        x21 = self.d21(torch.cat([self.ag["21"](f[3], f[2]), self.up(f[3])], 1))
        x31 = self.d31(torch.cat([self.ag["31"](f[4], f[3]), self.up(f[4])], 1))

        x02 = self.d02(torch.cat([f[0], self.ag["02"](x11, x01), self.up(x11)], 1))
        x12 = self.d12(torch.cat([f[1], self.ag["12"](x21, x11), self.up(x21)], 1))
        x22 = self.d22(torch.cat([f[2], self.ag["22"](x31, x21), self.up(x31)], 1))

        x03 = self.d03(torch.cat([f[0], x01, self.ag["03"](x12, x02), self.up(x12)], 1))
        x13 = self.d13(torch.cat([f[1], x11, self.ag["13"](x22, x12), self.up(x22)], 1))
        x04 = self.d04(torch.cat([f[0], x01, x02, self.ag["04"](x13, x03), self.up(x13)], 1))

        H, W = x.shape[2:]
        if self.deep_supervision:
            return [F.interpolate(h(o), size=(H, W), mode="bilinear", align_corners=True) for h, o in zip(self.heads, [x01, x02, x03, x04])]
        return F.interpolate(self.head(x04), size=(H, W), mode="bilinear", align_corners=True)

class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()): s.copy_(self.decay * s + (1 - self.decay) * m.data)
        for s, m in zip(self.shadow.buffers(), model.buffers()): s.copy_(m)
    def __call__(self, *args, **kwargs): return self.shadow(*args, **kwargs)

# =====================================================================
# 8. TRAINING SCRIPT
# =====================================================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    
    base = "dataset/train"
    img_dir = os.path.join(base, "images")
    msk_dir = os.path.join(base, "labels")
    if not os.path.exists(img_dir): img_dir = msk_dir = base

    all_ = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    random.shuffle(all_)
    split = int(0.8 * len(all_))
    tr_names, vl_names = all_[:split], all_[split:]

    tr_ds = DFUC2022Dataset(img_dir, msk_dir, tr_names, transform=get_train_transforms(), training=True)
    vl_ds = DFUC2022Dataset(img_dir, msk_dir, vl_names, transform=get_val_transforms(), training=False)
    
    tr_ld = DataLoader(tr_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    vl_ld = DataLoader(vl_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    model = MambaUNetPPv4(
        deep_supervision=CONFIG["use_deep_supervision"], 
        pretrained=HAS_TIMM
    ).to(device)

    params, flops, fps, inf_time = analyze_complexity(model, device)

    ema = EMA(model, decay=0.9995)
    crit = ScheduledLoss()

    enc_param_ids = {id(p) for p in model.encoder.parameters()}
    dec_params = [p for p in model.parameters() if id(p) not in enc_param_ids]

    optimizer = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()), "lr": 2e-5, "name": "encoder"},
        {"params": dec_params, "lr": 1e-4, "name": "decoder"},
    ], weight_decay=1e-4, betas=(0.9, 0.999))

    # FIXED: pct_start=0.3 gives the optimizer longer to warm up out of local minima
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[1e-4, 1e-3], epochs=CONFIG["epochs"], steps_per_epoch=len(tr_ld),
        pct_start=0.3, anneal_strategy="cos", div_factor=10, final_div_factor=100
    )

    scaler = torch.amp.GradScaler("cuda")
    ds_w = [0.05, 0.15, 0.30, 0.50]
    ACCUM = 2
    best_dice = 0.0
    patience_cnt = 0

    best_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0, "f1": 0}

    for epoch in range(CONFIG["epochs"]):
        model.train()
        crit.set_epoch(epoch)
        tloss = 0.0
        n_valid = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(enumerate(tr_ld), total=len(tr_ld), desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Train]", ncols=100)
        for step, (imgs, masks) in pbar:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            
            with torch.amp.autocast("cuda"):
                seg_outs = model(imgs)
                if isinstance(seg_outs, tuple): seg_outs = seg_outs[0]
                if not isinstance(seg_outs, list): seg_outs = [seg_outs]
                
                if CONFIG["use_deep_supervision"]:
                    loss = sum(w * crit(o, masks) for w, o in zip(ds_w[-len(seg_outs):], seg_outs)) / ACCUM
                else:
                    loss = crit(seg_outs[-1], masks) / ACCUM

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
            n_valid += 1

            if (step + 1) % ACCUM == 0 or (step + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
            sched.step()
            tloss += loss.item() * ACCUM

        ema.shadow.eval()
        d_l, i_l, p_l, r_l, f1_l = [], [], [], [], []
        
        with torch.no_grad():
            for imgs, masks in tqdm(vl_ld, desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Val]  ", leave=False, ncols=100):
                imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    out = ema.shadow(imgs)
                    final_out = out[-1] if isinstance(out, list) else out
                
                d, i, p, r, f1 = calc_metrics(final_out, masks)
                d_l.append(d); i_l.append(i); p_l.append(p); r_l.append(r); f1_l.append(f1)

        vd, vi, vp, vr, vf1 = np.mean(d_l), np.mean(i_l), np.mean(p_l), np.mean(r_l), np.mean(f1_l)
        print(f"  Dice:{vd:.4f} | IoU:{vi:.4f} | Prec:{vp:.4f} | Rec:{vr:.4f} | F1:{vf1:.4f}")

        if vd > best_dice:
            best_dice = vd
            best_metrics = {"dice": vd, "iou": vi, "precision": vp, "recall": vr, "f1": vf1}
            torch.save(ema.shadow.state_dict(), "best_model.pth")
            print(f"  ✓ Best model saved! (Dice={best_dice:.4f})")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= CONFIG["patience"]:
                print(f"\nEarly stop at epoch {epoch+1}.")
                break

    print("\n[Saving Metrics to metrics.csv...]")
    metrics_file = "metrics.csv"
    write_header = not os.path.exists(metrics_file)
    with open(metrics_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["Experiment", "Dice", "IoU", "Precision", "Recall", "F1", "Params_M", "FLOPs_G", "FPS", "Inference_ms"])
        
        exp_name = f"Mamba={CONFIG['use_mamba']}_DeepSup={CONFIG['use_deep_supervision']}_BiDir={CONFIG['bidirectional_mamba']}_LossSched={CONFIG['use_loss_schedule']}"
        writer.writerow([
            exp_name, f"{best_metrics['dice']:.4f}", f"{best_metrics['iou']:.4f}", f"{best_metrics['precision']:.4f}",
            f"{best_metrics['recall']:.4f}", f"{best_metrics['f1']:.4f}", f"{params/1e6:.2f}", f"{flops/1e9:.2f}",
            f"{fps:.2f}", f"{inf_time:.2f}"
        ])
    print("✓ Metrics successfully logged!")

if __name__ == "__main__":
    train()

✓ Fast CUDA Mamba loaded
✓ timm loaded — will use pretrained EfficientNet-B4 encoder

Device: cuda


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



[Profiling Hardware Complexity...]
  ➜ Params: 28.02 M
  ➜ FLOPs : 239.67 G
  ➜ FPS   : 36.2 imgs/sec (27.63 ms/img)



E01/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:58<00:00,  3.54it/s]
                                                                                                    

  Dice:0.0606 | IoU:0.0324 | Prec:0.0526 | Rec:0.1921 | F1:0.0606
  ✓ Best model saved! (Dice=0.0606)


E02/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.57it/s]
                                                                                                    

... (93 outputs left out)

E49/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [01:00<00:00,  3.44it/s]
                                                                                                    

  Dice:0.7758 | IoU:0.6905 | Prec:0.8961 | Rec:0.7750 | F1:0.7758


E50/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [01:00<00:00,  3.44it/s]
                                                                                                    

  Dice:0.7714 | IoU:0.6857 | Prec:0.9038 | Rec:0.7640 | F1:0.7714

[Saving Metrics to metrics.csv...]
✓ Metrics successfully logged!


In [5]:
# """
# Mamba-UNet++ v4 — DFUC2022 Edition (Targeting ~0.73+ SOTA Dice)
# ================================================================
# With Auto-Metrics, Complexity Analysis, and Zero-Collapse Fixes
# """

# ── ABLATION & TRAINING CONFIGURATION ────────────────────────────────
CONFIG = {
    "epochs": 50,
    "patience": 20,              # Less aggressive early stopping
    "use_mamba": False,           # False = replace Mamba with Identity
    "use_deep_supervision": True,# False = final output only
    "bidirectional_mamba": True, # False = disable reverse scan
    "use_loss_schedule": True    # False = Dice/BCE only
}
# ─────────────────────────────────────────────────────────────────────

import os

# Thread guards MUST come before cv2 / numpy import
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import math, random, warnings, csv, time
import cv2

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt
from copy import deepcopy

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── optional thop for FLOPs ──────────────────────────────────────────
try:
    from thop import profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("⚠  thop not found. Install via `pip install thop` for exact FLOPs.")

# ── optional fast Mamba ──────────────────────────────────────────────
try:
    from mamba_ssm import Mamba
    USE_FAST_MAMBA = True
    print("✓ Fast CUDA Mamba loaded")
except ImportError:
    USE_FAST_MAMBA = False
    print("⚠  mamba-ssm not found — using CBAM attention (fast fallback)")

# ── optional timm ────────────────────────────────────────────────────
try:
    import timm
    HAS_TIMM = True
    print("✓ timm loaded — will use pretrained EfficientNet-B4 encoder")
except ImportError:
    HAS_TIMM = False
    print("⚠  timm not found — falling back to scratch encoder")

# ── DFUC2022 constants ───────────────────────────────────────────────
TRAIN_PATCH_SIZE = 512
INFER_PATCH_SIZE = 512
INFER_STRIDE = 384
PAD_RESIZE = 640
IMG_SIZE = TRAIN_PATCH_SIZE
image_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_images'
mask_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_masks'

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()

# =====================================================================
# 1. COMPLEXITY ANALYSIS
# =====================================================================
def analyze_complexity(model, device, input_size=(1, 3, 512, 512)):
    print("\n[Profiling Hardware Complexity...]")
    dummy_input = torch.randn(input_size).to(device)
    model.eval()
    
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    flops = 0
    if HAS_THOP:
        with torch.no_grad():
            macs, _ = profile(deepcopy(model).to(device), inputs=(dummy_input,), verbose=False)
            flops = macs * 2
    
    runs = 100
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for _ in range(10): model(dummy_input)
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(runs): model(dummy_input)
        torch.cuda.synchronize()
        end_time = time.time()
        
    total_time = end_time - start_time
    time_per_img_ms = (total_time / runs) * 1000
    fps = 1000.0 / time_per_img_ms
    
    print(f"  ➜ Params: {params / 1e6:.2f} M")
    if HAS_THOP: print(f"  ➜ FLOPs : {flops / 1e9:.2f} G")
    print(f"  ➜ FPS   : {fps:.1f} imgs/sec ({time_per_img_ms:.2f} ms/img)\n")
    return params, flops, fps, time_per_img_ms

# =====================================================================
# 2. HELPERS
# =====================================================================
def pad_to_square(image, mask=None, pad_value=0):
    h, w = image.shape[:2]
    if h == w:
        if mask is not None: return image, mask, (0, 0, 0, 0)
        return image, (0, 0, 0, 0)
    diff = abs(h - w)
    pad_before = diff // 2
    pad_after = diff - pad_before
    top, bottom, left, right = (0, 0, pad_before, pad_after) if h > w else (pad_before, pad_after, 0, 0)
    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(pad_value, pad_value, pad_value))
    if mask is not None:
        mask = cv2.copyMakeBorder(mask, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)
        return image, mask, (top, bottom, left, right)
    return image, (top, bottom, left, right)

# =====================================================================
# 3. DATASET
# =====================================================================
class DFUC2022Dataset(Dataset):
    def __init__(self, image_dir, mask_dir, names, transform=None, training=True, patch_size=TRAIN_PATCH_SIZE, crops_per_image=1):
        self.transform = transform
        self.training = training
        self.patch_size = patch_size
        self.crops_per_image = crops_per_image
        self.samples = []
        for n in names:
            ip = os.path.join(image_dir, n)
            if not os.path.exists(ip) or os.path.getsize(ip) == 0: continue
            stem = n.rsplit(".", 1)[0]
            mp_png = os.path.join(mask_dir, stem + ".png")
            mp_jpg = os.path.join(mask_dir, n)
            mp = mp_png if os.path.exists(mp_png) else (mp_jpg if os.path.exists(mp_jpg) else None)
            self.samples.append({"img": ip, "mask": mp})

    def __len__(self):
        return len(self.samples) * self.crops_per_image if self.training else len(self.samples)

    @staticmethod
    def _bbox_of_mask(mask):
        ys, xs = np.where(mask > 0.5)
        if len(ys) == 0: return None
        return int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())

    def _random_crop_around_fg(self, image, mask):
        ps = self.patch_size
        h, w = image.shape[:2]
        pad_h, pad_w = max(ps - h, 0), max(ps - w, 0)
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=(0, 0, 0))
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            h, w = image.shape[:2]

        bbox = self._bbox_of_mask(mask)
        if bbox is not None:
            y0, y1, x0, x1 = bbox
            cy_lo, cy_hi = max(0, y1 - ps + 1), min(h - ps, y0)
            cx_lo, cx_hi = max(0, x1 - ps + 1), min(w - ps, x0)
            cy = random.randint(min(cy_lo, cy_hi), max(cy_lo, cy_hi))
            cx = random.randint(min(cx_lo, cx_hi), max(cx_lo, cx_hi))
        else:
            cy, cx = random.randint(0, h - ps), random.randint(0, w - ps)
        return image[cy : cy + ps, cx : cx + ps], mask[cy : cy + ps, cx : cx + ps]

    def __getitem__(self, idx):
        s = self.samples[idx % len(self.samples)]
        im = cv2.imread(s["img"])
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im is not None else np.zeros((TRAIN_PATCH_SIZE, TRAIN_PATCH_SIZE, 3), np.uint8)
        
        # FIXED: Absolute > 0 catch-all to prevent black-mask collapse
        mk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE) if s["mask"] else np.zeros(im.shape[:2], np.uint8)
        mk = (mk > 0).astype(np.float32)

        if self.training:
            im, mk = self._random_crop_around_fg(im, mk)
        else:
            im, mk, _ = pad_to_square(im, mk, pad_value=0)
            im = cv2.resize(im, (PAD_RESIZE, PAD_RESIZE))
            mk = cv2.resize(mk, (PAD_RESIZE, PAD_RESIZE), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            aug = self.transform(image=im, mask=mk)
            im, mk = aug["image"], aug["mask"].unsqueeze(0)
        return im, mk

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-30, 30), p=0.5),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05, p=0.2), A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.4), A.HueSaturationValue(8, 20, 10, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

# =====================================================================
# 4. METRICS
# =====================================================================
def calc_metrics(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float().cpu().numpy()
    tgt = target.float().cpu().numpy()
    dice_l, iou_l, prec_l, rec_l, f1_l = [], [], [], [], []
    for p, t in zip(pred, tgt):
        p, t = p.squeeze(), t.squeeze()
        tp = float((p * t).sum())
        fp = float((p * (1 - t)).sum())
        fn = float(((1 - p) * t).sum())
        dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
        dice_l.append(dice)
        iou_l.append((tp + 1e-6) / (tp + fp + fn + 1e-6))
        prec_l.append((tp + 1e-6) / (tp + fp + 1e-6))
        rec_l.append((tp + 1e-6) / (tp + fn + 1e-6))
        f1_l.append(dice)
    return map(np.mean, [dice_l, iou_l, prec_l, rec_l, f1_l])

# =====================================================================
# 5. LOSSES
# =====================================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce = self.bce(logits.float(), targets.float())
        p = torch.sigmoid(logits.float()).view(-1)
        t = targets.float().view(-1)
        dice = 1 - (2 * (p * t).sum() + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce, dice

class LovaszHingeLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits.float()).view(-1)
        tgts = targets.float().view(-1)
        if tgts.sum() == 0 or tgts.sum() == tgts.numel(): return (logits * 0.0).mean()
        errors = (tgts - probs).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        gts = tgts[perm].sum()
        intersection = gts - tgts[perm].cumsum(0)
        union = gts + (1 - tgts[perm]).cumsum(0)
        jaccard = 1.0 - intersection / (union + 1e-6)
        if tgts[perm].numel() > 1: jaccard[1:] = jaccard[1:] - jaccard[:-1]
        return torch.clamp(torch.dot(errors_sorted, jaccard), 0.0, 2.0)

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-3):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs.float()).view(-1)
        t = targets.float().view(-1)
        tp = (p * t).sum()
        fp = (p * (1 - t)).sum()
        fn = ((1 - p) * t).sum()
        tv = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        tv = torch.clamp(tv, min=1e-6, max=1.0)
        return (1 - tv) ** self.gamma

class ScheduledLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice_bce = DiceBCELoss()
        self.tversky = TverskyLoss()
        self.lovasz = LovaszHingeLoss()
        self._epoch = 0

    def set_epoch(self, e): self._epoch = e

    def forward(self, logits, targets):
        bce, dice = self.dice_bce(logits, targets)
        
        if not CONFIG["use_loss_schedule"]:
            return bce + dice
            
        # FIXED: Warm-up period. Pure BCE/Dice for 5 epochs to prevent Lovasz/Tversky NaN explosion
        if self._epoch < 5: 
            return bce + dice
            
        tv = self.tversky(logits, targets)
        lov = self.lovasz(logits, targets)
        def safe(t): return t if torch.isfinite(t) else torch.zeros(1, device=logits.device)[0]
        
        e = self._epoch
        if e < 20:   w = (0.25, 0.25, 0.40, 0.10)
        elif e < 50: w = (0.20, 0.30, 0.30, 0.20)
        else:        w = (0.15, 0.30, 0.25, 0.30)
        return w[0] * safe(bce) + w[1] * safe(dice) + w[2] * safe(tv) + w[3] * safe(lov)

# =====================================================================
# 6. BUILDING BLOCKS
# =====================================================================
class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.se(x).view(x.size(0), x.size(1), 1, 1)

class CBAM(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False))
        self.spatial = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, x):
        ca = torch.sigmoid(self.mlp(x.mean(dim=(2, 3))) + self.mlp(x.amax(dim=(2, 3)))).unsqueeze(-1).unsqueeze(-1)
        x = x * ca
        sa = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * self.spatial(sa)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, drop=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU(), nn.Dropout2d(drop),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.se = SEBlock(out_c)
        self.proj = nn.Conv2d(in_c, out_c, 1, bias=False) if in_c != out_c else nn.Identity()
    def forward(self, x): return self.se(self.conv(x)) + self.proj(x)

class ASPP(nn.Module):
    def __init__(self, in_c, out_c=256):
        super().__init__()
        rates = [1, 6, 12, 18]
        self.convs = nn.ModuleList([nn.Sequential(nn.Conv2d(in_c, out_c, 3 if r>1 else 1, padding=r if r>1 else 0, dilation=r, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)) for r in rates])
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_c, out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
        self.proj = nn.Sequential(nn.Conv2d(out_c * (len(rates) + 1), out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
    def forward(self, x):
        h, w = x.shape[2:]
        feats = [c(x) for c in self.convs] + [F.interpolate(self.pool(x), size=(h, w), mode="bilinear", align_corners=True)]
        return self.proj(torch.cat(feats, 1))

class AttentionGate(nn.Module):
    def __init__(self, Fg, Fl, Fi):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(Fg, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.Wx = nn.Sequential(nn.Conv2d(Fl, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        g1 = self.Wg(F.interpolate(g, x.shape[2:], mode="bilinear", align_corners=True))
        return x * self.psi(F.relu(g1 + self.Wx(x), inplace=True))

class FastMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mamba = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.norm = nn.LayerNorm(d)

    def _scan(self, x, flip=False):
        s = torch.flip(x, [1]) if flip else x
        o = self.mamba(self.norm(s))
        return torch.flip(o, [1]) if flip else o

    def forward(self, x):
        # Cast to float32 internally to prevent Mamba hidden state fp16 overflow
        with torch.amp.autocast("cuda", enabled=False):
            x_f32 = x.float()
            B, C, H, W = x_f32.shape
            h = x_f32.view(B, C, -1).transpose(1, 2)
            v = x_f32.permute(0, 1, 3, 2).contiguous().view(B, C, -1).transpose(1, 2)
            
            if CONFIG["bidirectional_mamba"]:
                oh = (self._scan(h) + self._scan(h, flip=True)) / 2
                ov = (self._scan(v) + self._scan(v, flip=True)) / 2
            else:
                oh = self._scan(h)
                ov = self._scan(v)
                
            out = ((oh + ov) / 2).transpose(1, 2).view(B, C, H, W) + x_f32
            return out.to(x.dtype)

# =====================================================================
# 7. ENCODER & UNET++
# =====================================================================
class ResidualAdapter(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.needs_proj = in_c != out_c
        if self.needs_proj:
            self.proj = nn.Conv2d(in_c, out_c, 1, bias=False)
            self.bn = nn.BatchNorm2d(out_c)
    def forward(self, x): return self.bn(self.proj(x)) if self.needs_proj else x

class EfficientNetEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.stages = timm.create_model("efficientnet_b4", pretrained=pretrained, features_only=True)
        self.channels = self.stages.feature_info.channels()
    def forward(self, x): return self.stages(x)

class MambaUNetPPv4(nn.Module):
    def __init__(self, num_classes=1, deep_supervision=True, pretrained=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.encoder = EfficientNetEncoder(pretrained=pretrained) if HAS_TIMM else ScratchEncoder()
        
        enc_ch = self.encoder.channels
        dec_ch = [64, 128, 192, 256, 256]
        self.adapt = nn.ModuleList([ResidualAdapter(ec, dc) for ec, dc in zip(enc_ch, dec_ch)])
        self.aspp = ASPP(dec_ch[4], dec_ch[4])
        
        if CONFIG["use_mamba"]:
            self.ctx3 = FastMambaBlock(dec_ch[3]) if USE_FAST_MAMBA else CBAM(dec_ch[3])
            self.ctx4 = FastMambaBlock(dec_ch[4]) if USE_FAST_MAMBA else CBAM(dec_ch[4])
        else:
            self.ctx3 = nn.Identity()
            self.ctx4 = nn.Identity()

        C = dec_ch
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag = nn.ModuleDict({
            "01": AttentionGate(C[1], C[0], C[0]//2), "11": AttentionGate(C[2], C[1], C[1]//2), "21": AttentionGate(C[3], C[2], C[2]//2), "31": AttentionGate(C[4], C[3], C[3]//2),
            "02": AttentionGate(C[1], C[0], C[0]//2), "12": AttentionGate(C[2], C[1], C[1]//2), "22": AttentionGate(C[3], C[2], C[2]//2),
            "03": AttentionGate(C[1], C[0], C[0]//2), "13": AttentionGate(C[2], C[1], C[1]//2), "04": AttentionGate(C[1], C[0], C[0]//2),
        })

        self.d01 = ConvBlock(C[0]+C[1], C[0])
        self.d11 = ConvBlock(C[1]+C[2], C[1])
        self.d21 = ConvBlock(C[2]+C[3], C[2])
        self.d31 = ConvBlock(C[3]+C[4], C[3])
        self.d02 = ConvBlock(C[0]*2+C[1], C[0])
        self.d12 = ConvBlock(C[1]*2+C[2], C[1])
        self.d22 = ConvBlock(C[2]*2+C[3], C[2])
        self.d03 = ConvBlock(C[0]*3+C[1], C[0])
        self.d13 = ConvBlock(C[1]*3+C[2], C[1])
        self.d04 = ConvBlock(C[0]*4+C[1], C[0])

        if deep_supervision: self.heads = nn.ModuleList([nn.Conv2d(C[0], num_classes, 1) for _ in range(4)])
        else: self.head = nn.Conv2d(C[0], num_classes, 1)
        
        # FIXED: Initialize decoder weights to prevent instant signal death
        self._init_weights()

    def _init_weights(self):
        for name, m in self.named_modules():
            if "encoder" not in name:
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        feats = self.encoder(x)
        f = [ad(fe) for ad, fe in zip(self.adapt, feats)]
        f[3] = self.ctx3(f[3])
        f[4] = self.ctx4(self.aspp(f[4]))

        x01 = self.d01(torch.cat([self.ag["01"](f[1], f[0]), self.up(f[1])], 1))
        x11 = self.d11(torch.cat([self.ag["11"](f[2], f[1]), self.up(f[2])], 1))
        x21 = self.d21(torch.cat([self.ag["21"](f[3], f[2]), self.up(f[3])], 1))
        x31 = self.d31(torch.cat([self.ag["31"](f[4], f[3]), self.up(f[4])], 1))

        x02 = self.d02(torch.cat([f[0], self.ag["02"](x11, x01), self.up(x11)], 1))
        x12 = self.d12(torch.cat([f[1], self.ag["12"](x21, x11), self.up(x21)], 1))
        x22 = self.d22(torch.cat([f[2], self.ag["22"](x31, x21), self.up(x31)], 1))

        x03 = self.d03(torch.cat([f[0], x01, self.ag["03"](x12, x02), self.up(x12)], 1))
        x13 = self.d13(torch.cat([f[1], x11, self.ag["13"](x22, x12), self.up(x22)], 1))
        x04 = self.d04(torch.cat([f[0], x01, x02, self.ag["04"](x13, x03), self.up(x13)], 1))

        H, W = x.shape[2:]
        if self.deep_supervision:
            return [F.interpolate(h(o), size=(H, W), mode="bilinear", align_corners=True) for h, o in zip(self.heads, [x01, x02, x03, x04])]
        return F.interpolate(self.head(x04), size=(H, W), mode="bilinear", align_corners=True)

class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()): s.copy_(self.decay * s + (1 - self.decay) * m.data)
        for s, m in zip(self.shadow.buffers(), model.buffers()): s.copy_(m)
    def __call__(self, *args, **kwargs): return self.shadow(*args, **kwargs)

# =====================================================================
# 8. TRAINING SCRIPT
# =====================================================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    
    base = "dataset/train"
    img_dir = os.path.join(base, "images")
    msk_dir = os.path.join(base, "labels")
    if not os.path.exists(img_dir): img_dir = msk_dir = base

    all_ = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    random.shuffle(all_)
    split = int(0.8 * len(all_))
    tr_names, vl_names = all_[:split], all_[split:]

    tr_ds = DFUC2022Dataset(img_dir, msk_dir, tr_names, transform=get_train_transforms(), training=True)
    vl_ds = DFUC2022Dataset(img_dir, msk_dir, vl_names, transform=get_val_transforms(), training=False)
    
    tr_ld = DataLoader(tr_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    vl_ld = DataLoader(vl_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    model = MambaUNetPPv4(
        deep_supervision=CONFIG["use_deep_supervision"], 
        pretrained=HAS_TIMM
    ).to(device)

    params, flops, fps, inf_time = analyze_complexity(model, device)

    ema = EMA(model, decay=0.9995)
    crit = ScheduledLoss()

    enc_param_ids = {id(p) for p in model.encoder.parameters()}
    dec_params = [p for p in model.parameters() if id(p) not in enc_param_ids]

    optimizer = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()), "lr": 2e-5, "name": "encoder"},
        {"params": dec_params, "lr": 1e-4, "name": "decoder"},
    ], weight_decay=1e-4, betas=(0.9, 0.999))

    # FIXED: pct_start=0.3 gives the optimizer longer to warm up out of local minima
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[1e-4, 1e-3], epochs=CONFIG["epochs"], steps_per_epoch=len(tr_ld),
        pct_start=0.3, anneal_strategy="cos", div_factor=10, final_div_factor=100
    )

    scaler = torch.amp.GradScaler("cuda")
    ds_w = [0.05, 0.15, 0.30, 0.50]
    ACCUM = 2
    best_dice = 0.0
    patience_cnt = 0

    best_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0, "f1": 0}

    for epoch in range(CONFIG["epochs"]):
        model.train()
        crit.set_epoch(epoch)
        tloss = 0.0
        n_valid = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(enumerate(tr_ld), total=len(tr_ld), desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Train]", ncols=100)
        for step, (imgs, masks) in pbar:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            
            with torch.amp.autocast("cuda"):
                seg_outs = model(imgs)
                if isinstance(seg_outs, tuple): seg_outs = seg_outs[0]
                if not isinstance(seg_outs, list): seg_outs = [seg_outs]
                
                if CONFIG["use_deep_supervision"]:
                    loss = sum(w * crit(o, masks) for w, o in zip(ds_w[-len(seg_outs):], seg_outs)) / ACCUM
                else:
                    loss = crit(seg_outs[-1], masks) / ACCUM

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
            n_valid += 1

            if (step + 1) % ACCUM == 0 or (step + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
            sched.step()
            tloss += loss.item() * ACCUM

        ema.shadow.eval()
        d_l, i_l, p_l, r_l, f1_l = [], [], [], [], []
        
        with torch.no_grad():
            for imgs, masks in tqdm(vl_ld, desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Val]  ", leave=False, ncols=100):
                imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    out = ema.shadow(imgs)
                    final_out = out[-1] if isinstance(out, list) else out
                
                d, i, p, r, f1 = calc_metrics(final_out, masks)
                d_l.append(d); i_l.append(i); p_l.append(p); r_l.append(r); f1_l.append(f1)

        vd, vi, vp, vr, vf1 = np.mean(d_l), np.mean(i_l), np.mean(p_l), np.mean(r_l), np.mean(f1_l)
        print(f"  Dice:{vd:.4f} | IoU:{vi:.4f} | Prec:{vp:.4f} | Rec:{vr:.4f} | F1:{vf1:.4f}")

        if vd > best_dice:
            best_dice = vd
            best_metrics = {"dice": vd, "iou": vi, "precision": vp, "recall": vr, "f1": vf1}
            torch.save(ema.shadow.state_dict(), "best_model.pth")
            print(f"  ✓ Best model saved! (Dice={best_dice:.4f})")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= CONFIG["patience"]:
                print(f"\nEarly stop at epoch {epoch+1}.")
                break

    print("\n[Saving Metrics to metrics.csv...]")
    metrics_file = "metrics.csv"
    write_header = not os.path.exists(metrics_file)
    with open(metrics_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["Experiment", "Dice", "IoU", "Precision", "Recall", "F1", "Params_M", "FLOPs_G", "FPS", "Inference_ms"])
        
        exp_name = f"Mamba={CONFIG['use_mamba']}_DeepSup={CONFIG['use_deep_supervision']}_BiDir={CONFIG['bidirectional_mamba']}_LossSched={CONFIG['use_loss_schedule']}"
        writer.writerow([
            exp_name, f"{best_metrics['dice']:.4f}", f"{best_metrics['iou']:.4f}", f"{best_metrics['precision']:.4f}",
            f"{best_metrics['recall']:.4f}", f"{best_metrics['f1']:.4f}", f"{params/1e6:.2f}", f"{flops/1e9:.2f}",
            f"{fps:.2f}", f"{inf_time:.2f}"
        ])
    print("✓ Metrics successfully logged!")

if __name__ == "__main__":
    train()

✓ Fast CUDA Mamba loaded
✓ timm loaded — will use pretrained EfficientNet-B4 encoder

Device: cuda


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



[Profiling Hardware Complexity...]
  ➜ Params: 27.15 M
  ➜ FLOPs : 238.04 G
  ➜ FPS   : 43.2 imgs/sec (23.15 ms/img)



E01/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:55<00:00,  3.72it/s]
                                                                                                    

  Dice:0.0974 | IoU:0.0545 | Prec:0.0658 | Rec:0.3849 | F1:0.0974
  ✓ Best model saved! (Dice=0.0974)


E02/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:55<00:00,  3.76it/s]
                                                                                                    

... (93 outputs left out)

E49/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:56<00:00,  3.67it/s]
                                                                                                    

  Dice:0.7667 | IoU:0.6755 | Prec:0.9066 | Rec:0.7490 | F1:0.7667
  ✓ Best model saved! (Dice=0.7667)


E50/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:56<00:00,  3.65it/s]
                                                                                                    

  Dice:0.7598 | IoU:0.6691 | Prec:0.9122 | Rec:0.7385 | F1:0.7598

[Saving Metrics to metrics.csv...]
✓ Metrics successfully logged!


In [6]:
# """
# Mamba-UNet++ v4 — DFUC2022 Edition (Targeting ~0.73+ SOTA Dice)
# ================================================================
# With Auto-Metrics, Complexity Analysis, and Zero-Collapse Fixes
# """

# ── ABLATION & TRAINING CONFIGURATION ────────────────────────────────
CONFIG = {
    "epochs": 50,
    "patience": 20,              # Less aggressive early stopping
    "use_mamba": True,           # False = replace Mamba with Identity
    "use_deep_supervision": False,# False = final output only
    "bidirectional_mamba": True, # False = disable reverse scan
    "use_loss_schedule": True    # False = Dice/BCE only
}
# ─────────────────────────────────────────────────────────────────────

import os

# Thread guards MUST come before cv2 / numpy import
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import math, random, warnings, csv, time
import cv2

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt
from copy import deepcopy

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── optional thop for FLOPs ──────────────────────────────────────────
try:
    from thop import profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("⚠  thop not found. Install via `pip install thop` for exact FLOPs.")

# ── optional fast Mamba ──────────────────────────────────────────────
try:
    from mamba_ssm import Mamba
    USE_FAST_MAMBA = True
    print("✓ Fast CUDA Mamba loaded")
except ImportError:
    USE_FAST_MAMBA = False
    print("⚠  mamba-ssm not found — using CBAM attention (fast fallback)")

# ── optional timm ────────────────────────────────────────────────────
try:
    import timm
    HAS_TIMM = True
    print("✓ timm loaded — will use pretrained EfficientNet-B4 encoder")
except ImportError:
    HAS_TIMM = False
    print("⚠  timm not found — falling back to scratch encoder")

# ── DFUC2022 constants ───────────────────────────────────────────────
TRAIN_PATCH_SIZE = 512
INFER_PATCH_SIZE = 512
INFER_STRIDE = 384
PAD_RESIZE = 640
IMG_SIZE = TRAIN_PATCH_SIZE
image_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_images'
mask_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_masks'

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()

# =====================================================================
# 1. COMPLEXITY ANALYSIS
# =====================================================================
def analyze_complexity(model, device, input_size=(1, 3, 512, 512)):
    print("\n[Profiling Hardware Complexity...]")
    dummy_input = torch.randn(input_size).to(device)
    model.eval()
    
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    flops = 0
    if HAS_THOP:
        with torch.no_grad():
            macs, _ = profile(deepcopy(model).to(device), inputs=(dummy_input,), verbose=False)
            flops = macs * 2
    
    runs = 100
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for _ in range(10): model(dummy_input)
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(runs): model(dummy_input)
        torch.cuda.synchronize()
        end_time = time.time()
        
    total_time = end_time - start_time
    time_per_img_ms = (total_time / runs) * 1000
    fps = 1000.0 / time_per_img_ms
    
    print(f"  ➜ Params: {params / 1e6:.2f} M")
    if HAS_THOP: print(f"  ➜ FLOPs : {flops / 1e9:.2f} G")
    print(f"  ➜ FPS   : {fps:.1f} imgs/sec ({time_per_img_ms:.2f} ms/img)\n")
    return params, flops, fps, time_per_img_ms

# =====================================================================
# 2. HELPERS
# =====================================================================
def pad_to_square(image, mask=None, pad_value=0):
    h, w = image.shape[:2]
    if h == w:
        if mask is not None: return image, mask, (0, 0, 0, 0)
        return image, (0, 0, 0, 0)
    diff = abs(h - w)
    pad_before = diff // 2
    pad_after = diff - pad_before
    top, bottom, left, right = (0, 0, pad_before, pad_after) if h > w else (pad_before, pad_after, 0, 0)
    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(pad_value, pad_value, pad_value))
    if mask is not None:
        mask = cv2.copyMakeBorder(mask, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)
        return image, mask, (top, bottom, left, right)
    return image, (top, bottom, left, right)

# =====================================================================
# 3. DATASET
# =====================================================================
class DFUC2022Dataset(Dataset):
    def __init__(self, image_dir, mask_dir, names, transform=None, training=True, patch_size=TRAIN_PATCH_SIZE, crops_per_image=1):
        self.transform = transform
        self.training = training
        self.patch_size = patch_size
        self.crops_per_image = crops_per_image
        self.samples = []
        for n in names:
            ip = os.path.join(image_dir, n)
            if not os.path.exists(ip) or os.path.getsize(ip) == 0: continue
            stem = n.rsplit(".", 1)[0]
            mp_png = os.path.join(mask_dir, stem + ".png")
            mp_jpg = os.path.join(mask_dir, n)
            mp = mp_png if os.path.exists(mp_png) else (mp_jpg if os.path.exists(mp_jpg) else None)
            self.samples.append({"img": ip, "mask": mp})

    def __len__(self):
        return len(self.samples) * self.crops_per_image if self.training else len(self.samples)

    @staticmethod
    def _bbox_of_mask(mask):
        ys, xs = np.where(mask > 0.5)
        if len(ys) == 0: return None
        return int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())

    def _random_crop_around_fg(self, image, mask):
        ps = self.patch_size
        h, w = image.shape[:2]
        pad_h, pad_w = max(ps - h, 0), max(ps - w, 0)
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=(0, 0, 0))
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            h, w = image.shape[:2]

        bbox = self._bbox_of_mask(mask)
        if bbox is not None:
            y0, y1, x0, x1 = bbox
            cy_lo, cy_hi = max(0, y1 - ps + 1), min(h - ps, y0)
            cx_lo, cx_hi = max(0, x1 - ps + 1), min(w - ps, x0)
            cy = random.randint(min(cy_lo, cy_hi), max(cy_lo, cy_hi))
            cx = random.randint(min(cx_lo, cx_hi), max(cx_lo, cx_hi))
        else:
            cy, cx = random.randint(0, h - ps), random.randint(0, w - ps)
        return image[cy : cy + ps, cx : cx + ps], mask[cy : cy + ps, cx : cx + ps]

    def __getitem__(self, idx):
        s = self.samples[idx % len(self.samples)]
        im = cv2.imread(s["img"])
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im is not None else np.zeros((TRAIN_PATCH_SIZE, TRAIN_PATCH_SIZE, 3), np.uint8)
        
        # FIXED: Absolute > 0 catch-all to prevent black-mask collapse
        mk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE) if s["mask"] else np.zeros(im.shape[:2], np.uint8)
        mk = (mk > 0).astype(np.float32)

        if self.training:
            im, mk = self._random_crop_around_fg(im, mk)
        else:
            im, mk, _ = pad_to_square(im, mk, pad_value=0)
            im = cv2.resize(im, (PAD_RESIZE, PAD_RESIZE))
            mk = cv2.resize(mk, (PAD_RESIZE, PAD_RESIZE), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            aug = self.transform(image=im, mask=mk)
            im, mk = aug["image"], aug["mask"].unsqueeze(0)
        return im, mk

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-30, 30), p=0.5),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05, p=0.2), A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.4), A.HueSaturationValue(8, 20, 10, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

# =====================================================================
# 4. METRICS
# =====================================================================
def calc_metrics(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float().cpu().numpy()
    tgt = target.float().cpu().numpy()
    dice_l, iou_l, prec_l, rec_l, f1_l = [], [], [], [], []
    for p, t in zip(pred, tgt):
        p, t = p.squeeze(), t.squeeze()
        tp = float((p * t).sum())
        fp = float((p * (1 - t)).sum())
        fn = float(((1 - p) * t).sum())
        dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
        dice_l.append(dice)
        iou_l.append((tp + 1e-6) / (tp + fp + fn + 1e-6))
        prec_l.append((tp + 1e-6) / (tp + fp + 1e-6))
        rec_l.append((tp + 1e-6) / (tp + fn + 1e-6))
        f1_l.append(dice)
    return map(np.mean, [dice_l, iou_l, prec_l, rec_l, f1_l])

# =====================================================================
# 5. LOSSES
# =====================================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce = self.bce(logits.float(), targets.float())
        p = torch.sigmoid(logits.float()).view(-1)
        t = targets.float().view(-1)
        dice = 1 - (2 * (p * t).sum() + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce, dice

class LovaszHingeLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits.float()).view(-1)
        tgts = targets.float().view(-1)
        if tgts.sum() == 0 or tgts.sum() == tgts.numel(): return (logits * 0.0).mean()
        errors = (tgts - probs).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        gts = tgts[perm].sum()
        intersection = gts - tgts[perm].cumsum(0)
        union = gts + (1 - tgts[perm]).cumsum(0)
        jaccard = 1.0 - intersection / (union + 1e-6)
        if tgts[perm].numel() > 1: jaccard[1:] = jaccard[1:] - jaccard[:-1]
        return torch.clamp(torch.dot(errors_sorted, jaccard), 0.0, 2.0)

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-3):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs.float()).view(-1)
        t = targets.float().view(-1)
        tp = (p * t).sum()
        fp = (p * (1 - t)).sum()
        fn = ((1 - p) * t).sum()
        tv = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        tv = torch.clamp(tv, min=1e-6, max=1.0)
        return (1 - tv) ** self.gamma

class ScheduledLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice_bce = DiceBCELoss()
        self.tversky = TverskyLoss()
        self.lovasz = LovaszHingeLoss()
        self._epoch = 0

    def set_epoch(self, e): self._epoch = e

    def forward(self, logits, targets):
        bce, dice = self.dice_bce(logits, targets)
        
        if not CONFIG["use_loss_schedule"]:
            return bce + dice
            
        # FIXED: Warm-up period. Pure BCE/Dice for 5 epochs to prevent Lovasz/Tversky NaN explosion
        if self._epoch < 5: 
            return bce + dice
            
        tv = self.tversky(logits, targets)
        lov = self.lovasz(logits, targets)
        def safe(t): return t if torch.isfinite(t) else torch.zeros(1, device=logits.device)[0]
        
        e = self._epoch
        if e < 20:   w = (0.25, 0.25, 0.40, 0.10)
        elif e < 50: w = (0.20, 0.30, 0.30, 0.20)
        else:        w = (0.15, 0.30, 0.25, 0.30)
        return w[0] * safe(bce) + w[1] * safe(dice) + w[2] * safe(tv) + w[3] * safe(lov)

# =====================================================================
# 6. BUILDING BLOCKS
# =====================================================================
class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.se(x).view(x.size(0), x.size(1), 1, 1)

class CBAM(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False))
        self.spatial = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, x):
        ca = torch.sigmoid(self.mlp(x.mean(dim=(2, 3))) + self.mlp(x.amax(dim=(2, 3)))).unsqueeze(-1).unsqueeze(-1)
        x = x * ca
        sa = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * self.spatial(sa)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, drop=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU(), nn.Dropout2d(drop),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.se = SEBlock(out_c)
        self.proj = nn.Conv2d(in_c, out_c, 1, bias=False) if in_c != out_c else nn.Identity()
    def forward(self, x): return self.se(self.conv(x)) + self.proj(x)

class ASPP(nn.Module):
    def __init__(self, in_c, out_c=256):
        super().__init__()
        rates = [1, 6, 12, 18]
        self.convs = nn.ModuleList([nn.Sequential(nn.Conv2d(in_c, out_c, 3 if r>1 else 1, padding=r if r>1 else 0, dilation=r, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)) for r in rates])
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_c, out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
        self.proj = nn.Sequential(nn.Conv2d(out_c * (len(rates) + 1), out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
    def forward(self, x):
        h, w = x.shape[2:]
        feats = [c(x) for c in self.convs] + [F.interpolate(self.pool(x), size=(h, w), mode="bilinear", align_corners=True)]
        return self.proj(torch.cat(feats, 1))

class AttentionGate(nn.Module):
    def __init__(self, Fg, Fl, Fi):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(Fg, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.Wx = nn.Sequential(nn.Conv2d(Fl, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        g1 = self.Wg(F.interpolate(g, x.shape[2:], mode="bilinear", align_corners=True))
        return x * self.psi(F.relu(g1 + self.Wx(x), inplace=True))

class FastMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mamba = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.norm = nn.LayerNorm(d)

    def _scan(self, x, flip=False):
        s = torch.flip(x, [1]) if flip else x
        o = self.mamba(self.norm(s))
        return torch.flip(o, [1]) if flip else o

    def forward(self, x):
        # Cast to float32 internally to prevent Mamba hidden state fp16 overflow
        with torch.amp.autocast("cuda", enabled=False):
            x_f32 = x.float()
            B, C, H, W = x_f32.shape
            h = x_f32.view(B, C, -1).transpose(1, 2)
            v = x_f32.permute(0, 1, 3, 2).contiguous().view(B, C, -1).transpose(1, 2)
            
            if CONFIG["bidirectional_mamba"]:
                oh = (self._scan(h) + self._scan(h, flip=True)) / 2
                ov = (self._scan(v) + self._scan(v, flip=True)) / 2
            else:
                oh = self._scan(h)
                ov = self._scan(v)
                
            out = ((oh + ov) / 2).transpose(1, 2).view(B, C, H, W) + x_f32
            return out.to(x.dtype)

# =====================================================================
# 7. ENCODER & UNET++
# =====================================================================
class ResidualAdapter(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.needs_proj = in_c != out_c
        if self.needs_proj:
            self.proj = nn.Conv2d(in_c, out_c, 1, bias=False)
            self.bn = nn.BatchNorm2d(out_c)
    def forward(self, x): return self.bn(self.proj(x)) if self.needs_proj else x

class EfficientNetEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.stages = timm.create_model("efficientnet_b4", pretrained=pretrained, features_only=True)
        self.channels = self.stages.feature_info.channels()
    def forward(self, x): return self.stages(x)

class MambaUNetPPv4(nn.Module):
    def __init__(self, num_classes=1, deep_supervision=True, pretrained=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.encoder = EfficientNetEncoder(pretrained=pretrained) if HAS_TIMM else ScratchEncoder()
        
        enc_ch = self.encoder.channels
        dec_ch = [64, 128, 192, 256, 256]
        self.adapt = nn.ModuleList([ResidualAdapter(ec, dc) for ec, dc in zip(enc_ch, dec_ch)])
        self.aspp = ASPP(dec_ch[4], dec_ch[4])
        
        if CONFIG["use_mamba"]:
            self.ctx3 = FastMambaBlock(dec_ch[3]) if USE_FAST_MAMBA else CBAM(dec_ch[3])
            self.ctx4 = FastMambaBlock(dec_ch[4]) if USE_FAST_MAMBA else CBAM(dec_ch[4])
        else:
            self.ctx3 = nn.Identity()
            self.ctx4 = nn.Identity()

        C = dec_ch
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag = nn.ModuleDict({
            "01": AttentionGate(C[1], C[0], C[0]//2), "11": AttentionGate(C[2], C[1], C[1]//2), "21": AttentionGate(C[3], C[2], C[2]//2), "31": AttentionGate(C[4], C[3], C[3]//2),
            "02": AttentionGate(C[1], C[0], C[0]//2), "12": AttentionGate(C[2], C[1], C[1]//2), "22": AttentionGate(C[3], C[2], C[2]//2),
            "03": AttentionGate(C[1], C[0], C[0]//2), "13": AttentionGate(C[2], C[1], C[1]//2), "04": AttentionGate(C[1], C[0], C[0]//2),
        })

        self.d01 = ConvBlock(C[0]+C[1], C[0])
        self.d11 = ConvBlock(C[1]+C[2], C[1])
        self.d21 = ConvBlock(C[2]+C[3], C[2])
        self.d31 = ConvBlock(C[3]+C[4], C[3])
        self.d02 = ConvBlock(C[0]*2+C[1], C[0])
        self.d12 = ConvBlock(C[1]*2+C[2], C[1])
        self.d22 = ConvBlock(C[2]*2+C[3], C[2])
        self.d03 = ConvBlock(C[0]*3+C[1], C[0])
        self.d13 = ConvBlock(C[1]*3+C[2], C[1])
        self.d04 = ConvBlock(C[0]*4+C[1], C[0])

        if deep_supervision: self.heads = nn.ModuleList([nn.Conv2d(C[0], num_classes, 1) for _ in range(4)])
        else: self.head = nn.Conv2d(C[0], num_classes, 1)
        
        # FIXED: Initialize decoder weights to prevent instant signal death
        self._init_weights()

    def _init_weights(self):
        for name, m in self.named_modules():
            if "encoder" not in name:
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        feats = self.encoder(x)
        f = [ad(fe) for ad, fe in zip(self.adapt, feats)]
        f[3] = self.ctx3(f[3])
        f[4] = self.ctx4(self.aspp(f[4]))

        x01 = self.d01(torch.cat([self.ag["01"](f[1], f[0]), self.up(f[1])], 1))
        x11 = self.d11(torch.cat([self.ag["11"](f[2], f[1]), self.up(f[2])], 1))
        x21 = self.d21(torch.cat([self.ag["21"](f[3], f[2]), self.up(f[3])], 1))
        x31 = self.d31(torch.cat([self.ag["31"](f[4], f[3]), self.up(f[4])], 1))

        x02 = self.d02(torch.cat([f[0], self.ag["02"](x11, x01), self.up(x11)], 1))
        x12 = self.d12(torch.cat([f[1], self.ag["12"](x21, x11), self.up(x21)], 1))
        x22 = self.d22(torch.cat([f[2], self.ag["22"](x31, x21), self.up(x31)], 1))

        x03 = self.d03(torch.cat([f[0], x01, self.ag["03"](x12, x02), self.up(x12)], 1))
        x13 = self.d13(torch.cat([f[1], x11, self.ag["13"](x22, x12), self.up(x22)], 1))
        x04 = self.d04(torch.cat([f[0], x01, x02, self.ag["04"](x13, x03), self.up(x13)], 1))

        H, W = x.shape[2:]
        if self.deep_supervision:
            return [F.interpolate(h(o), size=(H, W), mode="bilinear", align_corners=True) for h, o in zip(self.heads, [x01, x02, x03, x04])]
        return F.interpolate(self.head(x04), size=(H, W), mode="bilinear", align_corners=True)

class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()): s.copy_(self.decay * s + (1 - self.decay) * m.data)
        for s, m in zip(self.shadow.buffers(), model.buffers()): s.copy_(m)
    def __call__(self, *args, **kwargs): return self.shadow(*args, **kwargs)

# =====================================================================
# 8. TRAINING SCRIPT
# =====================================================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    
    base = "dataset/train"
    img_dir = os.path.join(base, "images")
    msk_dir = os.path.join(base, "labels")
    if not os.path.exists(img_dir): img_dir = msk_dir = base

    all_ = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    random.shuffle(all_)
    split = int(0.8 * len(all_))
    tr_names, vl_names = all_[:split], all_[split:]

    tr_ds = DFUC2022Dataset(img_dir, msk_dir, tr_names, transform=get_train_transforms(), training=True)
    vl_ds = DFUC2022Dataset(img_dir, msk_dir, vl_names, transform=get_val_transforms(), training=False)
    
    tr_ld = DataLoader(tr_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    vl_ld = DataLoader(vl_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    model = MambaUNetPPv4(
        deep_supervision=CONFIG["use_deep_supervision"], 
        pretrained=HAS_TIMM
    ).to(device)

    params, flops, fps, inf_time = analyze_complexity(model, device)

    ema = EMA(model, decay=0.9995)
    crit = ScheduledLoss()

    enc_param_ids = {id(p) for p in model.encoder.parameters()}
    dec_params = [p for p in model.parameters() if id(p) not in enc_param_ids]

    optimizer = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()), "lr": 2e-5, "name": "encoder"},
        {"params": dec_params, "lr": 1e-4, "name": "decoder"},
    ], weight_decay=1e-4, betas=(0.9, 0.999))

    # FIXED: pct_start=0.3 gives the optimizer longer to warm up out of local minima
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[1e-4, 1e-3], epochs=CONFIG["epochs"], steps_per_epoch=len(tr_ld),
        pct_start=0.3, anneal_strategy="cos", div_factor=10, final_div_factor=100
    )

    scaler = torch.amp.GradScaler("cuda")
    ds_w = [0.05, 0.15, 0.30, 0.50]
    ACCUM = 2
    best_dice = 0.0
    patience_cnt = 0

    best_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0, "f1": 0}

    for epoch in range(CONFIG["epochs"]):
        model.train()
        crit.set_epoch(epoch)
        tloss = 0.0
        n_valid = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(enumerate(tr_ld), total=len(tr_ld), desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Train]", ncols=100)
        for step, (imgs, masks) in pbar:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            
            with torch.amp.autocast("cuda"):
                seg_outs = model(imgs)
                if isinstance(seg_outs, tuple): seg_outs = seg_outs[0]
                if not isinstance(seg_outs, list): seg_outs = [seg_outs]
                
                if CONFIG["use_deep_supervision"]:
                    loss = sum(w * crit(o, masks) for w, o in zip(ds_w[-len(seg_outs):], seg_outs)) / ACCUM
                else:
                    loss = crit(seg_outs[-1], masks) / ACCUM

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
            n_valid += 1

            if (step + 1) % ACCUM == 0 or (step + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
            sched.step()
            tloss += loss.item() * ACCUM

        ema.shadow.eval()
        d_l, i_l, p_l, r_l, f1_l = [], [], [], [], []
        
        with torch.no_grad():
            for imgs, masks in tqdm(vl_ld, desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Val]  ", leave=False, ncols=100):
                imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    out = ema.shadow(imgs)
                    final_out = out[-1] if isinstance(out, list) else out
                
                d, i, p, r, f1 = calc_metrics(final_out, masks)
                d_l.append(d); i_l.append(i); p_l.append(p); r_l.append(r); f1_l.append(f1)

        vd, vi, vp, vr, vf1 = np.mean(d_l), np.mean(i_l), np.mean(p_l), np.mean(r_l), np.mean(f1_l)
        print(f"  Dice:{vd:.4f} | IoU:{vi:.4f} | Prec:{vp:.4f} | Rec:{vr:.4f} | F1:{vf1:.4f}")

        if vd > best_dice:
            best_dice = vd
            best_metrics = {"dice": vd, "iou": vi, "precision": vp, "recall": vr, "f1": vf1}
            torch.save(ema.shadow.state_dict(), "best_model.pth")
            print(f"  ✓ Best model saved! (Dice={best_dice:.4f})")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= CONFIG["patience"]:
                print(f"\nEarly stop at epoch {epoch+1}.")
                break

    print("\n[Saving Metrics to metrics.csv...]")
    metrics_file = "metrics.csv"
    write_header = not os.path.exists(metrics_file)
    with open(metrics_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["Experiment", "Dice", "IoU", "Precision", "Recall", "F1", "Params_M", "FLOPs_G", "FPS", "Inference_ms"])
        
        exp_name = f"Mamba={CONFIG['use_mamba']}_DeepSup={CONFIG['use_deep_supervision']}_BiDir={CONFIG['bidirectional_mamba']}_LossSched={CONFIG['use_loss_schedule']}"
        writer.writerow([
            exp_name, f"{best_metrics['dice']:.4f}", f"{best_metrics['iou']:.4f}", f"{best_metrics['precision']:.4f}",
            f"{best_metrics['recall']:.4f}", f"{best_metrics['f1']:.4f}", f"{params/1e6:.2f}", f"{flops/1e9:.2f}",
            f"{fps:.2f}", f"{inf_time:.2f}"
        ])
    print("✓ Metrics successfully logged!")

if __name__ == "__main__":
    train()

✓ Fast CUDA Mamba loaded
✓ timm loaded — will use pretrained EfficientNet-B4 encoder

Device: cuda


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



[Profiling Hardware Complexity...]
  ➜ Params: 28.02 M
  ➜ FLOPs : 239.64 G
  ➜ FPS   : 37.1 imgs/sec (26.97 ms/img)



E01/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:56<00:00,  3.66it/s]
                                                                                                    

  Dice:0.1752 | IoU:0.1053 | Prec:0.1208 | Rec:0.7913 | F1:0.1752
  ✓ Best model saved! (Dice=0.1752)


E02/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.61it/s]
                                                                                                    

... (93 outputs left out)

E49/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.58it/s]
                                                                                                    

  Dice:0.7662 | IoU:0.6722 | Prec:0.9107 | Rec:0.7435 | F1:0.7662


E50/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.58it/s]
                                                                                                    

  Dice:0.7520 | IoU:0.6601 | Prec:0.9149 | Rec:0.7277 | F1:0.7520

[Saving Metrics to metrics.csv...]
✓ Metrics successfully logged!


In [7]:
# """
# Mamba-UNet++ v4 — DFUC2022 Edition (Targeting ~0.73+ SOTA Dice)
# ================================================================
# With Auto-Metrics, Complexity Analysis, and Zero-Collapse Fixes
# """

# ── ABLATION & TRAINING CONFIGURATION ────────────────────────────────
CONFIG = {
    "epochs": 50,
    "patience": 20,              # Less aggressive early stopping
    "use_mamba": True,           # False = replace Mamba with Identity
    "use_deep_supervision": True,# False = final output only
    "bidirectional_mamba": False, # False = disable reverse scan
    "use_loss_schedule": True    # False = Dice/BCE only
}
# ─────────────────────────────────────────────────────────────────────

import os

# Thread guards MUST come before cv2 / numpy import
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import math, random, warnings, csv, time
import cv2

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt
from copy import deepcopy

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── optional thop for FLOPs ──────────────────────────────────────────
try:
    from thop import profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("⚠  thop not found. Install via `pip install thop` for exact FLOPs.")

# ── optional fast Mamba ──────────────────────────────────────────────
try:
    from mamba_ssm import Mamba
    USE_FAST_MAMBA = True
    print("✓ Fast CUDA Mamba loaded")
except ImportError:
    USE_FAST_MAMBA = False
    print("⚠  mamba-ssm not found — using CBAM attention (fast fallback)")

# ── optional timm ────────────────────────────────────────────────────
try:
    import timm
    HAS_TIMM = True
    print("✓ timm loaded — will use pretrained EfficientNet-B4 encoder")
except ImportError:
    HAS_TIMM = False
    print("⚠  timm not found — falling back to scratch encoder")

# ── DFUC2022 constants ───────────────────────────────────────────────
TRAIN_PATCH_SIZE = 512
INFER_PATCH_SIZE = 512
INFER_STRIDE = 384
PAD_RESIZE = 640
IMG_SIZE = TRAIN_PATCH_SIZE
image_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_images'
mask_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_masks'

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()

# =====================================================================
# 1. COMPLEXITY ANALYSIS
# =====================================================================
def analyze_complexity(model, device, input_size=(1, 3, 512, 512)):
    print("\n[Profiling Hardware Complexity...]")
    dummy_input = torch.randn(input_size).to(device)
    model.eval()
    
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    flops = 0
    if HAS_THOP:
        with torch.no_grad():
            macs, _ = profile(deepcopy(model).to(device), inputs=(dummy_input,), verbose=False)
            flops = macs * 2
    
    runs = 100
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for _ in range(10): model(dummy_input)
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(runs): model(dummy_input)
        torch.cuda.synchronize()
        end_time = time.time()
        
    total_time = end_time - start_time
    time_per_img_ms = (total_time / runs) * 1000
    fps = 1000.0 / time_per_img_ms
    
    print(f"  ➜ Params: {params / 1e6:.2f} M")
    if HAS_THOP: print(f"  ➜ FLOPs : {flops / 1e9:.2f} G")
    print(f"  ➜ FPS   : {fps:.1f} imgs/sec ({time_per_img_ms:.2f} ms/img)\n")
    return params, flops, fps, time_per_img_ms

# =====================================================================
# 2. HELPERS
# =====================================================================
def pad_to_square(image, mask=None, pad_value=0):
    h, w = image.shape[:2]
    if h == w:
        if mask is not None: return image, mask, (0, 0, 0, 0)
        return image, (0, 0, 0, 0)
    diff = abs(h - w)
    pad_before = diff // 2
    pad_after = diff - pad_before
    top, bottom, left, right = (0, 0, pad_before, pad_after) if h > w else (pad_before, pad_after, 0, 0)
    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(pad_value, pad_value, pad_value))
    if mask is not None:
        mask = cv2.copyMakeBorder(mask, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)
        return image, mask, (top, bottom, left, right)
    return image, (top, bottom, left, right)

# =====================================================================
# 3. DATASET
# =====================================================================
class DFUC2022Dataset(Dataset):
    def __init__(self, image_dir, mask_dir, names, transform=None, training=True, patch_size=TRAIN_PATCH_SIZE, crops_per_image=1):
        self.transform = transform
        self.training = training
        self.patch_size = patch_size
        self.crops_per_image = crops_per_image
        self.samples = []
        for n in names:
            ip = os.path.join(image_dir, n)
            if not os.path.exists(ip) or os.path.getsize(ip) == 0: continue
            stem = n.rsplit(".", 1)[0]
            mp_png = os.path.join(mask_dir, stem + ".png")
            mp_jpg = os.path.join(mask_dir, n)
            mp = mp_png if os.path.exists(mp_png) else (mp_jpg if os.path.exists(mp_jpg) else None)
            self.samples.append({"img": ip, "mask": mp})

    def __len__(self):
        return len(self.samples) * self.crops_per_image if self.training else len(self.samples)

    @staticmethod
    def _bbox_of_mask(mask):
        ys, xs = np.where(mask > 0.5)
        if len(ys) == 0: return None
        return int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())

    def _random_crop_around_fg(self, image, mask):
        ps = self.patch_size
        h, w = image.shape[:2]
        pad_h, pad_w = max(ps - h, 0), max(ps - w, 0)
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=(0, 0, 0))
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            h, w = image.shape[:2]

        bbox = self._bbox_of_mask(mask)
        if bbox is not None:
            y0, y1, x0, x1 = bbox
            cy_lo, cy_hi = max(0, y1 - ps + 1), min(h - ps, y0)
            cx_lo, cx_hi = max(0, x1 - ps + 1), min(w - ps, x0)
            cy = random.randint(min(cy_lo, cy_hi), max(cy_lo, cy_hi))
            cx = random.randint(min(cx_lo, cx_hi), max(cx_lo, cx_hi))
        else:
            cy, cx = random.randint(0, h - ps), random.randint(0, w - ps)
        return image[cy : cy + ps, cx : cx + ps], mask[cy : cy + ps, cx : cx + ps]

    def __getitem__(self, idx):
        s = self.samples[idx % len(self.samples)]
        im = cv2.imread(s["img"])
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im is not None else np.zeros((TRAIN_PATCH_SIZE, TRAIN_PATCH_SIZE, 3), np.uint8)
        
        # FIXED: Absolute > 0 catch-all to prevent black-mask collapse
        mk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE) if s["mask"] else np.zeros(im.shape[:2], np.uint8)
        mk = (mk > 0).astype(np.float32)

        if self.training:
            im, mk = self._random_crop_around_fg(im, mk)
        else:
            im, mk, _ = pad_to_square(im, mk, pad_value=0)
            im = cv2.resize(im, (PAD_RESIZE, PAD_RESIZE))
            mk = cv2.resize(mk, (PAD_RESIZE, PAD_RESIZE), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            aug = self.transform(image=im, mask=mk)
            im, mk = aug["image"], aug["mask"].unsqueeze(0)
        return im, mk

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-30, 30), p=0.5),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05, p=0.2), A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.4), A.HueSaturationValue(8, 20, 10, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

# =====================================================================
# 4. METRICS
# =====================================================================
def calc_metrics(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float().cpu().numpy()
    tgt = target.float().cpu().numpy()
    dice_l, iou_l, prec_l, rec_l, f1_l = [], [], [], [], []
    for p, t in zip(pred, tgt):
        p, t = p.squeeze(), t.squeeze()
        tp = float((p * t).sum())
        fp = float((p * (1 - t)).sum())
        fn = float(((1 - p) * t).sum())
        dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
        dice_l.append(dice)
        iou_l.append((tp + 1e-6) / (tp + fp + fn + 1e-6))
        prec_l.append((tp + 1e-6) / (tp + fp + 1e-6))
        rec_l.append((tp + 1e-6) / (tp + fn + 1e-6))
        f1_l.append(dice)
    return map(np.mean, [dice_l, iou_l, prec_l, rec_l, f1_l])

# =====================================================================
# 5. LOSSES
# =====================================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce = self.bce(logits.float(), targets.float())
        p = torch.sigmoid(logits.float()).view(-1)
        t = targets.float().view(-1)
        dice = 1 - (2 * (p * t).sum() + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce, dice

class LovaszHingeLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits.float()).view(-1)
        tgts = targets.float().view(-1)
        if tgts.sum() == 0 or tgts.sum() == tgts.numel(): return (logits * 0.0).mean()
        errors = (tgts - probs).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        gts = tgts[perm].sum()
        intersection = gts - tgts[perm].cumsum(0)
        union = gts + (1 - tgts[perm]).cumsum(0)
        jaccard = 1.0 - intersection / (union + 1e-6)
        if tgts[perm].numel() > 1: jaccard[1:] = jaccard[1:] - jaccard[:-1]
        return torch.clamp(torch.dot(errors_sorted, jaccard), 0.0, 2.0)

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-3):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs.float()).view(-1)
        t = targets.float().view(-1)
        tp = (p * t).sum()
        fp = (p * (1 - t)).sum()
        fn = ((1 - p) * t).sum()
        tv = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        tv = torch.clamp(tv, min=1e-6, max=1.0)
        return (1 - tv) ** self.gamma

class ScheduledLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice_bce = DiceBCELoss()
        self.tversky = TverskyLoss()
        self.lovasz = LovaszHingeLoss()
        self._epoch = 0

    def set_epoch(self, e): self._epoch = e

    def forward(self, logits, targets):
        bce, dice = self.dice_bce(logits, targets)
        
        if not CONFIG["use_loss_schedule"]:
            return bce + dice
            
        # FIXED: Warm-up period. Pure BCE/Dice for 5 epochs to prevent Lovasz/Tversky NaN explosion
        if self._epoch < 5: 
            return bce + dice
            
        tv = self.tversky(logits, targets)
        lov = self.lovasz(logits, targets)
        def safe(t): return t if torch.isfinite(t) else torch.zeros(1, device=logits.device)[0]
        
        e = self._epoch
        if e < 20:   w = (0.25, 0.25, 0.40, 0.10)
        elif e < 50: w = (0.20, 0.30, 0.30, 0.20)
        else:        w = (0.15, 0.30, 0.25, 0.30)
        return w[0] * safe(bce) + w[1] * safe(dice) + w[2] * safe(tv) + w[3] * safe(lov)

# =====================================================================
# 6. BUILDING BLOCKS
# =====================================================================
class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.se(x).view(x.size(0), x.size(1), 1, 1)

class CBAM(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False))
        self.spatial = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, x):
        ca = torch.sigmoid(self.mlp(x.mean(dim=(2, 3))) + self.mlp(x.amax(dim=(2, 3)))).unsqueeze(-1).unsqueeze(-1)
        x = x * ca
        sa = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * self.spatial(sa)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, drop=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU(), nn.Dropout2d(drop),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.se = SEBlock(out_c)
        self.proj = nn.Conv2d(in_c, out_c, 1, bias=False) if in_c != out_c else nn.Identity()
    def forward(self, x): return self.se(self.conv(x)) + self.proj(x)

class ASPP(nn.Module):
    def __init__(self, in_c, out_c=256):
        super().__init__()
        rates = [1, 6, 12, 18]
        self.convs = nn.ModuleList([nn.Sequential(nn.Conv2d(in_c, out_c, 3 if r>1 else 1, padding=r if r>1 else 0, dilation=r, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)) for r in rates])
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_c, out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
        self.proj = nn.Sequential(nn.Conv2d(out_c * (len(rates) + 1), out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
    def forward(self, x):
        h, w = x.shape[2:]
        feats = [c(x) for c in self.convs] + [F.interpolate(self.pool(x), size=(h, w), mode="bilinear", align_corners=True)]
        return self.proj(torch.cat(feats, 1))

class AttentionGate(nn.Module):
    def __init__(self, Fg, Fl, Fi):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(Fg, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.Wx = nn.Sequential(nn.Conv2d(Fl, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        g1 = self.Wg(F.interpolate(g, x.shape[2:], mode="bilinear", align_corners=True))
        return x * self.psi(F.relu(g1 + self.Wx(x), inplace=True))

class FastMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mamba = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.norm = nn.LayerNorm(d)

    def _scan(self, x, flip=False):
        s = torch.flip(x, [1]) if flip else x
        o = self.mamba(self.norm(s))
        return torch.flip(o, [1]) if flip else o

    def forward(self, x):
        # Cast to float32 internally to prevent Mamba hidden state fp16 overflow
        with torch.amp.autocast("cuda", enabled=False):
            x_f32 = x.float()
            B, C, H, W = x_f32.shape
            h = x_f32.view(B, C, -1).transpose(1, 2)
            v = x_f32.permute(0, 1, 3, 2).contiguous().view(B, C, -1).transpose(1, 2)
            
            if CONFIG["bidirectional_mamba"]:
                oh = (self._scan(h) + self._scan(h, flip=True)) / 2
                ov = (self._scan(v) + self._scan(v, flip=True)) / 2
            else:
                oh = self._scan(h)
                ov = self._scan(v)
                
            out = ((oh + ov) / 2).transpose(1, 2).view(B, C, H, W) + x_f32
            return out.to(x.dtype)

# =====================================================================
# 7. ENCODER & UNET++
# =====================================================================
class ResidualAdapter(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.needs_proj = in_c != out_c
        if self.needs_proj:
            self.proj = nn.Conv2d(in_c, out_c, 1, bias=False)
            self.bn = nn.BatchNorm2d(out_c)
    def forward(self, x): return self.bn(self.proj(x)) if self.needs_proj else x

class EfficientNetEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.stages = timm.create_model("efficientnet_b4", pretrained=pretrained, features_only=True)
        self.channels = self.stages.feature_info.channels()
    def forward(self, x): return self.stages(x)

class MambaUNetPPv4(nn.Module):
    def __init__(self, num_classes=1, deep_supervision=True, pretrained=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.encoder = EfficientNetEncoder(pretrained=pretrained) if HAS_TIMM else ScratchEncoder()
        
        enc_ch = self.encoder.channels
        dec_ch = [64, 128, 192, 256, 256]
        self.adapt = nn.ModuleList([ResidualAdapter(ec, dc) for ec, dc in zip(enc_ch, dec_ch)])
        self.aspp = ASPP(dec_ch[4], dec_ch[4])
        
        if CONFIG["use_mamba"]:
            self.ctx3 = FastMambaBlock(dec_ch[3]) if USE_FAST_MAMBA else CBAM(dec_ch[3])
            self.ctx4 = FastMambaBlock(dec_ch[4]) if USE_FAST_MAMBA else CBAM(dec_ch[4])
        else:
            self.ctx3 = nn.Identity()
            self.ctx4 = nn.Identity()

        C = dec_ch
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag = nn.ModuleDict({
            "01": AttentionGate(C[1], C[0], C[0]//2), "11": AttentionGate(C[2], C[1], C[1]//2), "21": AttentionGate(C[3], C[2], C[2]//2), "31": AttentionGate(C[4], C[3], C[3]//2),
            "02": AttentionGate(C[1], C[0], C[0]//2), "12": AttentionGate(C[2], C[1], C[1]//2), "22": AttentionGate(C[3], C[2], C[2]//2),
            "03": AttentionGate(C[1], C[0], C[0]//2), "13": AttentionGate(C[2], C[1], C[1]//2), "04": AttentionGate(C[1], C[0], C[0]//2),
        })

        self.d01 = ConvBlock(C[0]+C[1], C[0])
        self.d11 = ConvBlock(C[1]+C[2], C[1])
        self.d21 = ConvBlock(C[2]+C[3], C[2])
        self.d31 = ConvBlock(C[3]+C[4], C[3])
        self.d02 = ConvBlock(C[0]*2+C[1], C[0])
        self.d12 = ConvBlock(C[1]*2+C[2], C[1])
        self.d22 = ConvBlock(C[2]*2+C[3], C[2])
        self.d03 = ConvBlock(C[0]*3+C[1], C[0])
        self.d13 = ConvBlock(C[1]*3+C[2], C[1])
        self.d04 = ConvBlock(C[0]*4+C[1], C[0])

        if deep_supervision: self.heads = nn.ModuleList([nn.Conv2d(C[0], num_classes, 1) for _ in range(4)])
        else: self.head = nn.Conv2d(C[0], num_classes, 1)
        
        # FIXED: Initialize decoder weights to prevent instant signal death
        self._init_weights()

    def _init_weights(self):
        for name, m in self.named_modules():
            if "encoder" not in name:
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        feats = self.encoder(x)
        f = [ad(fe) for ad, fe in zip(self.adapt, feats)]
        f[3] = self.ctx3(f[3])
        f[4] = self.ctx4(self.aspp(f[4]))

        x01 = self.d01(torch.cat([self.ag["01"](f[1], f[0]), self.up(f[1])], 1))
        x11 = self.d11(torch.cat([self.ag["11"](f[2], f[1]), self.up(f[2])], 1))
        x21 = self.d21(torch.cat([self.ag["21"](f[3], f[2]), self.up(f[3])], 1))
        x31 = self.d31(torch.cat([self.ag["31"](f[4], f[3]), self.up(f[4])], 1))

        x02 = self.d02(torch.cat([f[0], self.ag["02"](x11, x01), self.up(x11)], 1))
        x12 = self.d12(torch.cat([f[1], self.ag["12"](x21, x11), self.up(x21)], 1))
        x22 = self.d22(torch.cat([f[2], self.ag["22"](x31, x21), self.up(x31)], 1))

        x03 = self.d03(torch.cat([f[0], x01, self.ag["03"](x12, x02), self.up(x12)], 1))
        x13 = self.d13(torch.cat([f[1], x11, self.ag["13"](x22, x12), self.up(x22)], 1))
        x04 = self.d04(torch.cat([f[0], x01, x02, self.ag["04"](x13, x03), self.up(x13)], 1))

        H, W = x.shape[2:]
        if self.deep_supervision:
            return [F.interpolate(h(o), size=(H, W), mode="bilinear", align_corners=True) for h, o in zip(self.heads, [x01, x02, x03, x04])]
        return F.interpolate(self.head(x04), size=(H, W), mode="bilinear", align_corners=True)

class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()): s.copy_(self.decay * s + (1 - self.decay) * m.data)
        for s, m in zip(self.shadow.buffers(), model.buffers()): s.copy_(m)
    def __call__(self, *args, **kwargs): return self.shadow(*args, **kwargs)

# =====================================================================
# 8. TRAINING SCRIPT
# =====================================================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    
    base = "dataset/train"
    img_dir = os.path.join(base, "images")
    msk_dir = os.path.join(base, "labels")
    if not os.path.exists(img_dir): img_dir = msk_dir = base

    all_ = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    random.shuffle(all_)
    split = int(0.8 * len(all_))
    tr_names, vl_names = all_[:split], all_[split:]

    tr_ds = DFUC2022Dataset(img_dir, msk_dir, tr_names, transform=get_train_transforms(), training=True)
    vl_ds = DFUC2022Dataset(img_dir, msk_dir, vl_names, transform=get_val_transforms(), training=False)
    
    tr_ld = DataLoader(tr_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    vl_ld = DataLoader(vl_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    model = MambaUNetPPv4(
        deep_supervision=CONFIG["use_deep_supervision"], 
        pretrained=HAS_TIMM
    ).to(device)

    params, flops, fps, inf_time = analyze_complexity(model, device)

    ema = EMA(model, decay=0.9995)
    crit = ScheduledLoss()

    enc_param_ids = {id(p) for p in model.encoder.parameters()}
    dec_params = [p for p in model.parameters() if id(p) not in enc_param_ids]

    optimizer = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()), "lr": 2e-5, "name": "encoder"},
        {"params": dec_params, "lr": 1e-4, "name": "decoder"},
    ], weight_decay=1e-4, betas=(0.9, 0.999))

    # FIXED: pct_start=0.3 gives the optimizer longer to warm up out of local minima
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[1e-4, 1e-3], epochs=CONFIG["epochs"], steps_per_epoch=len(tr_ld),
        pct_start=0.3, anneal_strategy="cos", div_factor=10, final_div_factor=100
    )

    scaler = torch.amp.GradScaler("cuda")
    ds_w = [0.05, 0.15, 0.30, 0.50]
    ACCUM = 2
    best_dice = 0.0
    patience_cnt = 0

    best_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0, "f1": 0}

    for epoch in range(CONFIG["epochs"]):
        model.train()
        crit.set_epoch(epoch)
        tloss = 0.0
        n_valid = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(enumerate(tr_ld), total=len(tr_ld), desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Train]", ncols=100)
        for step, (imgs, masks) in pbar:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            
            with torch.amp.autocast("cuda"):
                seg_outs = model(imgs)
                if isinstance(seg_outs, tuple): seg_outs = seg_outs[0]
                if not isinstance(seg_outs, list): seg_outs = [seg_outs]
                
                if CONFIG["use_deep_supervision"]:
                    loss = sum(w * crit(o, masks) for w, o in zip(ds_w[-len(seg_outs):], seg_outs)) / ACCUM
                else:
                    loss = crit(seg_outs[-1], masks) / ACCUM

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
            n_valid += 1

            if (step + 1) % ACCUM == 0 or (step + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
            sched.step()
            tloss += loss.item() * ACCUM

        ema.shadow.eval()
        d_l, i_l, p_l, r_l, f1_l = [], [], [], [], []
        
        with torch.no_grad():
            for imgs, masks in tqdm(vl_ld, desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Val]  ", leave=False, ncols=100):
                imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    out = ema.shadow(imgs)
                    final_out = out[-1] if isinstance(out, list) else out
                
                d, i, p, r, f1 = calc_metrics(final_out, masks)
                d_l.append(d); i_l.append(i); p_l.append(p); r_l.append(r); f1_l.append(f1)

        vd, vi, vp, vr, vf1 = np.mean(d_l), np.mean(i_l), np.mean(p_l), np.mean(r_l), np.mean(f1_l)
        print(f"  Dice:{vd:.4f} | IoU:{vi:.4f} | Prec:{vp:.4f} | Rec:{vr:.4f} | F1:{vf1:.4f}")

        if vd > best_dice:
            best_dice = vd
            best_metrics = {"dice": vd, "iou": vi, "precision": vp, "recall": vr, "f1": vf1}
            torch.save(ema.shadow.state_dict(), "best_model.pth")
            print(f"  ✓ Best model saved! (Dice={best_dice:.4f})")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= CONFIG["patience"]:
                print(f"\nEarly stop at epoch {epoch+1}.")
                break

    print("\n[Saving Metrics to metrics.csv...]")
    metrics_file = "metrics.csv"
    write_header = not os.path.exists(metrics_file)
    with open(metrics_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["Experiment", "Dice", "IoU", "Precision", "Recall", "F1", "Params_M", "FLOPs_G", "FPS", "Inference_ms"])
        
        exp_name = f"Mamba={CONFIG['use_mamba']}_DeepSup={CONFIG['use_deep_supervision']}_BiDir={CONFIG['bidirectional_mamba']}_LossSched={CONFIG['use_loss_schedule']}"
        writer.writerow([
            exp_name, f"{best_metrics['dice']:.4f}", f"{best_metrics['iou']:.4f}", f"{best_metrics['precision']:.4f}",
            f"{best_metrics['recall']:.4f}", f"{best_metrics['f1']:.4f}", f"{params/1e6:.2f}", f"{flops/1e9:.2f}",
            f"{fps:.2f}", f"{inf_time:.2f}"
        ])
    print("✓ Metrics successfully logged!")

if __name__ == "__main__":
    train()

✓ Fast CUDA Mamba loaded
✓ timm loaded — will use pretrained EfficientNet-B4 encoder

Device: cuda


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



[Profiling Hardware Complexity...]
  ➜ Params: 28.02 M
  ➜ FLOPs : 238.86 G
  ➜ FPS   : 39.6 imgs/sec (25.26 ms/img)



E01/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:55<00:00,  3.74it/s]
                                                                                                    

  Dice:0.0643 | IoU:0.0345 | Prec:0.0517 | Rec:0.2059 | F1:0.0643
  ✓ Best model saved! (Dice=0.0643)


E02/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:55<00:00,  3.72it/s]
                                                                                                    

... (93 outputs left out)

E49/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.62it/s]
                                                                                                    

  Dice:0.7974 | IoU:0.7110 | Prec:0.8868 | Rec:0.8028 | F1:0.7974
  ✓ Best model saved! (Dice=0.7974)


E50/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.59it/s]
                                                                                                    

  Dice:0.7940 | IoU:0.7081 | Prec:0.8948 | Rec:0.7918 | F1:0.7940

[Saving Metrics to metrics.csv...]
✓ Metrics successfully logged!


In [8]:
# """
# Mamba-UNet++ v4 — DFUC2022 Edition (Targeting ~0.73+ SOTA Dice)
# ================================================================
# With Auto-Metrics, Complexity Analysis, and Zero-Collapse Fixes
# """

# ── ABLATION & TRAINING CONFIGURATION ────────────────────────────────
CONFIG = {
    "epochs": 50,
    "patience": 20,              # Less aggressive early stopping
    "use_mamba": True,           # False = replace Mamba with Identity
    "use_deep_supervision": True,# False = final output only
    "bidirectional_mamba": True, # False = disable reverse scan
    "use_loss_schedule": False    # False = Dice/BCE only
}
# ─────────────────────────────────────────────────────────────────────

import os

# Thread guards MUST come before cv2 / numpy import
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import math, random, warnings, csv, time
import cv2

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt
from copy import deepcopy

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── optional thop for FLOPs ──────────────────────────────────────────
try:
    from thop import profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("⚠  thop not found. Install via `pip install thop` for exact FLOPs.")

# ── optional fast Mamba ──────────────────────────────────────────────
try:
    from mamba_ssm import Mamba
    USE_FAST_MAMBA = True
    print("✓ Fast CUDA Mamba loaded")
except ImportError:
    USE_FAST_MAMBA = False
    print("⚠  mamba-ssm not found — using CBAM attention (fast fallback)")

# ── optional timm ────────────────────────────────────────────────────
try:
    import timm
    HAS_TIMM = True
    print("✓ timm loaded — will use pretrained EfficientNet-B4 encoder")
except ImportError:
    HAS_TIMM = False
    print("⚠  timm not found — falling back to scratch encoder")

# ── DFUC2022 constants ───────────────────────────────────────────────
TRAIN_PATCH_SIZE = 512
INFER_PATCH_SIZE = 512
INFER_STRIDE = 384
PAD_RESIZE = 640
IMG_SIZE = TRAIN_PATCH_SIZE
image_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_images'
mask_dir = 'dataset/DFUC2022_train_release/DFUC2022_train_masks'

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()

# =====================================================================
# 1. COMPLEXITY ANALYSIS
# =====================================================================
def analyze_complexity(model, device, input_size=(1, 3, 512, 512)):
    print("\n[Profiling Hardware Complexity...]")
    dummy_input = torch.randn(input_size).to(device)
    model.eval()
    
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    flops = 0
    if HAS_THOP:
        with torch.no_grad():
            macs, _ = profile(deepcopy(model).to(device), inputs=(dummy_input,), verbose=False)
            flops = macs * 2
    
    runs = 100
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for _ in range(10): model(dummy_input)
        torch.cuda.synchronize()
        start_time = time.time()
        for _ in range(runs): model(dummy_input)
        torch.cuda.synchronize()
        end_time = time.time()
        
    total_time = end_time - start_time
    time_per_img_ms = (total_time / runs) * 1000
    fps = 1000.0 / time_per_img_ms
    
    print(f"  ➜ Params: {params / 1e6:.2f} M")
    if HAS_THOP: print(f"  ➜ FLOPs : {flops / 1e9:.2f} G")
    print(f"  ➜ FPS   : {fps:.1f} imgs/sec ({time_per_img_ms:.2f} ms/img)\n")
    return params, flops, fps, time_per_img_ms

# =====================================================================
# 2. HELPERS
# =====================================================================
def pad_to_square(image, mask=None, pad_value=0):
    h, w = image.shape[:2]
    if h == w:
        if mask is not None: return image, mask, (0, 0, 0, 0)
        return image, (0, 0, 0, 0)
    diff = abs(h - w)
    pad_before = diff // 2
    pad_after = diff - pad_before
    top, bottom, left, right = (0, 0, pad_before, pad_after) if h > w else (pad_before, pad_after, 0, 0)
    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(pad_value, pad_value, pad_value))
    if mask is not None:
        mask = cv2.copyMakeBorder(mask, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)
        return image, mask, (top, bottom, left, right)
    return image, (top, bottom, left, right)

# =====================================================================
# 3. DATASET
# =====================================================================
class DFUC2022Dataset(Dataset):
    def __init__(self, image_dir, mask_dir, names, transform=None, training=True, patch_size=TRAIN_PATCH_SIZE, crops_per_image=1):
        self.transform = transform
        self.training = training
        self.patch_size = patch_size
        self.crops_per_image = crops_per_image
        self.samples = []
        for n in names:
            ip = os.path.join(image_dir, n)
            if not os.path.exists(ip) or os.path.getsize(ip) == 0: continue
            stem = n.rsplit(".", 1)[0]
            mp_png = os.path.join(mask_dir, stem + ".png")
            mp_jpg = os.path.join(mask_dir, n)
            mp = mp_png if os.path.exists(mp_png) else (mp_jpg if os.path.exists(mp_jpg) else None)
            self.samples.append({"img": ip, "mask": mp})

    def __len__(self):
        return len(self.samples) * self.crops_per_image if self.training else len(self.samples)

    @staticmethod
    def _bbox_of_mask(mask):
        ys, xs = np.where(mask > 0.5)
        if len(ys) == 0: return None
        return int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())

    def _random_crop_around_fg(self, image, mask):
        ps = self.patch_size
        h, w = image.shape[:2]
        pad_h, pad_w = max(ps - h, 0), max(ps - w, 0)
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=(0, 0, 0))
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            h, w = image.shape[:2]

        bbox = self._bbox_of_mask(mask)
        if bbox is not None:
            y0, y1, x0, x1 = bbox
            cy_lo, cy_hi = max(0, y1 - ps + 1), min(h - ps, y0)
            cx_lo, cx_hi = max(0, x1 - ps + 1), min(w - ps, x0)
            cy = random.randint(min(cy_lo, cy_hi), max(cy_lo, cy_hi))
            cx = random.randint(min(cx_lo, cx_hi), max(cx_lo, cx_hi))
        else:
            cy, cx = random.randint(0, h - ps), random.randint(0, w - ps)
        return image[cy : cy + ps, cx : cx + ps], mask[cy : cy + ps, cx : cx + ps]

    def __getitem__(self, idx):
        s = self.samples[idx % len(self.samples)]
        im = cv2.imread(s["img"])
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im is not None else np.zeros((TRAIN_PATCH_SIZE, TRAIN_PATCH_SIZE, 3), np.uint8)
        
        # FIXED: Absolute > 0 catch-all to prevent black-mask collapse
        mk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE) if s["mask"] else np.zeros(im.shape[:2], np.uint8)
        mk = (mk > 0).astype(np.float32)

        if self.training:
            im, mk = self._random_crop_around_fg(im, mk)
        else:
            im, mk, _ = pad_to_square(im, mk, pad_value=0)
            im = cv2.resize(im, (PAD_RESIZE, PAD_RESIZE))
            mk = cv2.resize(mk, (PAD_RESIZE, PAD_RESIZE), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            aug = self.transform(image=im, mask=mk)
            im, mk = aug["image"], aug["mask"].unsqueeze(0)
        return im, mk

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-30, 30), p=0.5),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05, p=0.2), A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.4), A.HueSaturationValue(8, 20, 10, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

# =====================================================================
# 4. METRICS
# =====================================================================
def calc_metrics(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float().cpu().numpy()
    tgt = target.float().cpu().numpy()
    dice_l, iou_l, prec_l, rec_l, f1_l = [], [], [], [], []
    for p, t in zip(pred, tgt):
        p, t = p.squeeze(), t.squeeze()
        tp = float((p * t).sum())
        fp = float((p * (1 - t)).sum())
        fn = float(((1 - p) * t).sum())
        dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
        dice_l.append(dice)
        iou_l.append((tp + 1e-6) / (tp + fp + fn + 1e-6))
        prec_l.append((tp + 1e-6) / (tp + fp + 1e-6))
        rec_l.append((tp + 1e-6) / (tp + fn + 1e-6))
        f1_l.append(dice)
    return map(np.mean, [dice_l, iou_l, prec_l, rec_l, f1_l])

# =====================================================================
# 5. LOSSES
# =====================================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce = self.bce(logits.float(), targets.float())
        p = torch.sigmoid(logits.float()).view(-1)
        t = targets.float().view(-1)
        dice = 1 - (2 * (p * t).sum() + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce, dice

class LovaszHingeLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits.float()).view(-1)
        tgts = targets.float().view(-1)
        if tgts.sum() == 0 or tgts.sum() == tgts.numel(): return (logits * 0.0).mean()
        errors = (tgts - probs).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        gts = tgts[perm].sum()
        intersection = gts - tgts[perm].cumsum(0)
        union = gts + (1 - tgts[perm]).cumsum(0)
        jaccard = 1.0 - intersection / (union + 1e-6)
        if tgts[perm].numel() > 1: jaccard[1:] = jaccard[1:] - jaccard[:-1]
        return torch.clamp(torch.dot(errors_sorted, jaccard), 0.0, 2.0)

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-3):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs.float()).view(-1)
        t = targets.float().view(-1)
        tp = (p * t).sum()
        fp = (p * (1 - t)).sum()
        fn = ((1 - p) * t).sum()
        tv = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        tv = torch.clamp(tv, min=1e-6, max=1.0)
        return (1 - tv) ** self.gamma

class ScheduledLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice_bce = DiceBCELoss()
        self.tversky = TverskyLoss()
        self.lovasz = LovaszHingeLoss()
        self._epoch = 0

    def set_epoch(self, e): self._epoch = e

    def forward(self, logits, targets):
        bce, dice = self.dice_bce(logits, targets)
        
        if not CONFIG["use_loss_schedule"]:
            return bce + dice
            
        # FIXED: Warm-up period. Pure BCE/Dice for 5 epochs to prevent Lovasz/Tversky NaN explosion
        if self._epoch < 5: 
            return bce + dice
            
        tv = self.tversky(logits, targets)
        lov = self.lovasz(logits, targets)
        def safe(t): return t if torch.isfinite(t) else torch.zeros(1, device=logits.device)[0]
        
        e = self._epoch
        if e < 20:   w = (0.25, 0.25, 0.40, 0.10)
        elif e < 50: w = (0.20, 0.30, 0.30, 0.20)
        else:        w = (0.15, 0.30, 0.25, 0.30)
        return w[0] * safe(bce) + w[1] * safe(dice) + w[2] * safe(tv) + w[3] * safe(lov)

# =====================================================================
# 6. BUILDING BLOCKS
# =====================================================================
class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.se(x).view(x.size(0), x.size(1), 1, 1)

class CBAM(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True), nn.Linear(c // r, c, bias=False))
        self.spatial = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, x):
        ca = torch.sigmoid(self.mlp(x.mean(dim=(2, 3))) + self.mlp(x.amax(dim=(2, 3)))).unsqueeze(-1).unsqueeze(-1)
        x = x * ca
        sa = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * self.spatial(sa)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, drop=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU(), nn.Dropout2d(drop),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.GELU()
        )
        self.se = SEBlock(out_c)
        self.proj = nn.Conv2d(in_c, out_c, 1, bias=False) if in_c != out_c else nn.Identity()
    def forward(self, x): return self.se(self.conv(x)) + self.proj(x)

class ASPP(nn.Module):
    def __init__(self, in_c, out_c=256):
        super().__init__()
        rates = [1, 6, 12, 18]
        self.convs = nn.ModuleList([nn.Sequential(nn.Conv2d(in_c, out_c, 3 if r>1 else 1, padding=r if r>1 else 0, dilation=r, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)) for r in rates])
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_c, out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
        self.proj = nn.Sequential(nn.Conv2d(out_c * (len(rates) + 1), out_c, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
    def forward(self, x):
        h, w = x.shape[2:]
        feats = [c(x) for c in self.convs] + [F.interpolate(self.pool(x), size=(h, w), mode="bilinear", align_corners=True)]
        return self.proj(torch.cat(feats, 1))

class AttentionGate(nn.Module):
    def __init__(self, Fg, Fl, Fi):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(Fg, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.Wx = nn.Sequential(nn.Conv2d(Fl, Fi, 1, bias=False), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        g1 = self.Wg(F.interpolate(g, x.shape[2:], mode="bilinear", align_corners=True))
        return x * self.psi(F.relu(g1 + self.Wx(x), inplace=True))

class FastMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mamba = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.norm = nn.LayerNorm(d)

    def _scan(self, x, flip=False):
        s = torch.flip(x, [1]) if flip else x
        o = self.mamba(self.norm(s))
        return torch.flip(o, [1]) if flip else o

    def forward(self, x):
        # Cast to float32 internally to prevent Mamba hidden state fp16 overflow
        with torch.amp.autocast("cuda", enabled=False):
            x_f32 = x.float()
            B, C, H, W = x_f32.shape
            h = x_f32.view(B, C, -1).transpose(1, 2)
            v = x_f32.permute(0, 1, 3, 2).contiguous().view(B, C, -1).transpose(1, 2)
            
            if CONFIG["bidirectional_mamba"]:
                oh = (self._scan(h) + self._scan(h, flip=True)) / 2
                ov = (self._scan(v) + self._scan(v, flip=True)) / 2
            else:
                oh = self._scan(h)
                ov = self._scan(v)
                
            out = ((oh + ov) / 2).transpose(1, 2).view(B, C, H, W) + x_f32
            return out.to(x.dtype)

# =====================================================================
# 7. ENCODER & UNET++
# =====================================================================
class ResidualAdapter(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.needs_proj = in_c != out_c
        if self.needs_proj:
            self.proj = nn.Conv2d(in_c, out_c, 1, bias=False)
            self.bn = nn.BatchNorm2d(out_c)
    def forward(self, x): return self.bn(self.proj(x)) if self.needs_proj else x

class EfficientNetEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.stages = timm.create_model("efficientnet_b4", pretrained=pretrained, features_only=True)
        self.channels = self.stages.feature_info.channels()
    def forward(self, x): return self.stages(x)

class MambaUNetPPv4(nn.Module):
    def __init__(self, num_classes=1, deep_supervision=True, pretrained=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.encoder = EfficientNetEncoder(pretrained=pretrained) if HAS_TIMM else ScratchEncoder()
        
        enc_ch = self.encoder.channels
        dec_ch = [64, 128, 192, 256, 256]
        self.adapt = nn.ModuleList([ResidualAdapter(ec, dc) for ec, dc in zip(enc_ch, dec_ch)])
        self.aspp = ASPP(dec_ch[4], dec_ch[4])
        
        if CONFIG["use_mamba"]:
            self.ctx3 = FastMambaBlock(dec_ch[3]) if USE_FAST_MAMBA else CBAM(dec_ch[3])
            self.ctx4 = FastMambaBlock(dec_ch[4]) if USE_FAST_MAMBA else CBAM(dec_ch[4])
        else:
            self.ctx3 = nn.Identity()
            self.ctx4 = nn.Identity()

        C = dec_ch
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag = nn.ModuleDict({
            "01": AttentionGate(C[1], C[0], C[0]//2), "11": AttentionGate(C[2], C[1], C[1]//2), "21": AttentionGate(C[3], C[2], C[2]//2), "31": AttentionGate(C[4], C[3], C[3]//2),
            "02": AttentionGate(C[1], C[0], C[0]//2), "12": AttentionGate(C[2], C[1], C[1]//2), "22": AttentionGate(C[3], C[2], C[2]//2),
            "03": AttentionGate(C[1], C[0], C[0]//2), "13": AttentionGate(C[2], C[1], C[1]//2), "04": AttentionGate(C[1], C[0], C[0]//2),
        })

        self.d01 = ConvBlock(C[0]+C[1], C[0])
        self.d11 = ConvBlock(C[1]+C[2], C[1])
        self.d21 = ConvBlock(C[2]+C[3], C[2])
        self.d31 = ConvBlock(C[3]+C[4], C[3])
        self.d02 = ConvBlock(C[0]*2+C[1], C[0])
        self.d12 = ConvBlock(C[1]*2+C[2], C[1])
        self.d22 = ConvBlock(C[2]*2+C[3], C[2])
        self.d03 = ConvBlock(C[0]*3+C[1], C[0])
        self.d13 = ConvBlock(C[1]*3+C[2], C[1])
        self.d04 = ConvBlock(C[0]*4+C[1], C[0])

        if deep_supervision: self.heads = nn.ModuleList([nn.Conv2d(C[0], num_classes, 1) for _ in range(4)])
        else: self.head = nn.Conv2d(C[0], num_classes, 1)
        
        # FIXED: Initialize decoder weights to prevent instant signal death
        self._init_weights()

    def _init_weights(self):
        for name, m in self.named_modules():
            if "encoder" not in name:
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        feats = self.encoder(x)
        f = [ad(fe) for ad, fe in zip(self.adapt, feats)]
        f[3] = self.ctx3(f[3])
        f[4] = self.ctx4(self.aspp(f[4]))

        x01 = self.d01(torch.cat([self.ag["01"](f[1], f[0]), self.up(f[1])], 1))
        x11 = self.d11(torch.cat([self.ag["11"](f[2], f[1]), self.up(f[2])], 1))
        x21 = self.d21(torch.cat([self.ag["21"](f[3], f[2]), self.up(f[3])], 1))
        x31 = self.d31(torch.cat([self.ag["31"](f[4], f[3]), self.up(f[4])], 1))

        x02 = self.d02(torch.cat([f[0], self.ag["02"](x11, x01), self.up(x11)], 1))
        x12 = self.d12(torch.cat([f[1], self.ag["12"](x21, x11), self.up(x21)], 1))
        x22 = self.d22(torch.cat([f[2], self.ag["22"](x31, x21), self.up(x31)], 1))

        x03 = self.d03(torch.cat([f[0], x01, self.ag["03"](x12, x02), self.up(x12)], 1))
        x13 = self.d13(torch.cat([f[1], x11, self.ag["13"](x22, x12), self.up(x22)], 1))
        x04 = self.d04(torch.cat([f[0], x01, x02, self.ag["04"](x13, x03), self.up(x13)], 1))

        H, W = x.shape[2:]
        if self.deep_supervision:
            return [F.interpolate(h(o), size=(H, W), mode="bilinear", align_corners=True) for h, o in zip(self.heads, [x01, x02, x03, x04])]
        return F.interpolate(self.head(x04), size=(H, W), mode="bilinear", align_corners=True)

class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()): s.copy_(self.decay * s + (1 - self.decay) * m.data)
        for s, m in zip(self.shadow.buffers(), model.buffers()): s.copy_(m)
    def __call__(self, *args, **kwargs): return self.shadow(*args, **kwargs)

# =====================================================================
# 8. TRAINING SCRIPT
# =====================================================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    
    base = "dataset/train"
    img_dir = os.path.join(base, "images")
    msk_dir = os.path.join(base, "labels")
    if not os.path.exists(img_dir): img_dir = msk_dir = base

    all_ = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    random.shuffle(all_)
    split = int(0.8 * len(all_))
    tr_names, vl_names = all_[:split], all_[split:]

    tr_ds = DFUC2022Dataset(img_dir, msk_dir, tr_names, transform=get_train_transforms(), training=True)
    vl_ds = DFUC2022Dataset(img_dir, msk_dir, vl_names, transform=get_val_transforms(), training=False)
    
    tr_ld = DataLoader(tr_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    vl_ld = DataLoader(vl_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    model = MambaUNetPPv4(
        deep_supervision=CONFIG["use_deep_supervision"], 
        pretrained=HAS_TIMM
    ).to(device)

    params, flops, fps, inf_time = analyze_complexity(model, device)

    ema = EMA(model, decay=0.9995)
    crit = ScheduledLoss()

    enc_param_ids = {id(p) for p in model.encoder.parameters()}
    dec_params = [p for p in model.parameters() if id(p) not in enc_param_ids]

    optimizer = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()), "lr": 2e-5, "name": "encoder"},
        {"params": dec_params, "lr": 1e-4, "name": "decoder"},
    ], weight_decay=1e-4, betas=(0.9, 0.999))

    # FIXED: pct_start=0.3 gives the optimizer longer to warm up out of local minima
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[1e-4, 1e-3], epochs=CONFIG["epochs"], steps_per_epoch=len(tr_ld),
        pct_start=0.3, anneal_strategy="cos", div_factor=10, final_div_factor=100
    )

    scaler = torch.amp.GradScaler("cuda")
    ds_w = [0.05, 0.15, 0.30, 0.50]
    ACCUM = 2
    best_dice = 0.0
    patience_cnt = 0

    best_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0, "f1": 0}

    for epoch in range(CONFIG["epochs"]):
        model.train()
        crit.set_epoch(epoch)
        tloss = 0.0
        n_valid = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(enumerate(tr_ld), total=len(tr_ld), desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Train]", ncols=100)
        for step, (imgs, masks) in pbar:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            
            with torch.amp.autocast("cuda"):
                seg_outs = model(imgs)
                if isinstance(seg_outs, tuple): seg_outs = seg_outs[0]
                if not isinstance(seg_outs, list): seg_outs = [seg_outs]
                
                if CONFIG["use_deep_supervision"]:
                    loss = sum(w * crit(o, masks) for w, o in zip(ds_w[-len(seg_outs):], seg_outs)) / ACCUM
                else:
                    loss = crit(seg_outs[-1], masks) / ACCUM

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
            n_valid += 1

            if (step + 1) % ACCUM == 0 or (step + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
            sched.step()
            tloss += loss.item() * ACCUM

        ema.shadow.eval()
        d_l, i_l, p_l, r_l, f1_l = [], [], [], [], []
        
        with torch.no_grad():
            for imgs, masks in tqdm(vl_ld, desc=f"E{epoch+1:02d}/{CONFIG['epochs']} [Val]  ", leave=False, ncols=100):
                imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    out = ema.shadow(imgs)
                    final_out = out[-1] if isinstance(out, list) else out
                
                d, i, p, r, f1 = calc_metrics(final_out, masks)
                d_l.append(d); i_l.append(i); p_l.append(p); r_l.append(r); f1_l.append(f1)

        vd, vi, vp, vr, vf1 = np.mean(d_l), np.mean(i_l), np.mean(p_l), np.mean(r_l), np.mean(f1_l)
        print(f"  Dice:{vd:.4f} | IoU:{vi:.4f} | Prec:{vp:.4f} | Rec:{vr:.4f} | F1:{vf1:.4f}")

        if vd > best_dice:
            best_dice = vd
            best_metrics = {"dice": vd, "iou": vi, "precision": vp, "recall": vr, "f1": vf1}
            torch.save(ema.shadow.state_dict(), "best_model.pth")
            print(f"  ✓ Best model saved! (Dice={best_dice:.4f})")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= CONFIG["patience"]:
                print(f"\nEarly stop at epoch {epoch+1}.")
                break

    print("\n[Saving Metrics to metrics.csv...]")
    metrics_file = "metrics.csv"
    write_header = not os.path.exists(metrics_file)
    with open(metrics_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["Experiment", "Dice", "IoU", "Precision", "Recall", "F1", "Params_M", "FLOPs_G", "FPS", "Inference_ms"])
        
        exp_name = f"Mamba={CONFIG['use_mamba']}_DeepSup={CONFIG['use_deep_supervision']}_BiDir={CONFIG['bidirectional_mamba']}_LossSched={CONFIG['use_loss_schedule']}"
        writer.writerow([
            exp_name, f"{best_metrics['dice']:.4f}", f"{best_metrics['iou']:.4f}", f"{best_metrics['precision']:.4f}",
            f"{best_metrics['recall']:.4f}", f"{best_metrics['f1']:.4f}", f"{params/1e6:.2f}", f"{flops/1e9:.2f}",
            f"{fps:.2f}", f"{inf_time:.2f}"
        ])
    print("✓ Metrics successfully logged!")

if __name__ == "__main__":
    train()

✓ Fast CUDA Mamba loaded
✓ timm loaded — will use pretrained EfficientNet-B4 encoder

Device: cuda


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



[Profiling Hardware Complexity...]
  ➜ Params: 28.02 M
  ➜ FLOPs : 239.67 G
  ➜ FPS   : 37.6 imgs/sec (26.57 ms/img)



E01/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.62it/s]
                                                                                                    

  Dice:0.0714 | IoU:0.0386 | Prec:0.0543 | Rec:0.2493 | F1:0.0714
  ✓ Best model saved! (Dice=0.0714)


E02/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.57it/s]
                                                                                                    

... (93 outputs left out)

E49/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:58<00:00,  3.56it/s]
                                                                                                    

  Dice:0.8052 | IoU:0.7196 | Prec:0.8612 | Rec:0.8369 | F1:0.8052
  ✓ Best model saved! (Dice=0.8052)


E50/50 [Train]: 100%|█████████████████████████████████████████████| 207/207 [00:57<00:00,  3.59it/s]
                                                                                                    

  Dice:0.8002 | IoU:0.7152 | Prec:0.8711 | Rec:0.8235 | F1:0.8002

[Saving Metrics to metrics.csv...]
✓ Metrics successfully logged!
